In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

try:
    ROOT = Path(__file__).resolve().parent.parent
except NameError:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_PATH = ROOT / "dataset" / "kosovo_overall_biodiversity.csv"
REPORTS_DIR = ROOT / "reports"


def section(title):
    print(f"\n{'=' * 70}\n{title}\n{'=' * 70}")

## 1. MBLEDHJA E TË DHËNAVE

In [2]:
section("1. MBLEDHJA E TË DHËNAVE")
df_raw = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Rreshta: {df_raw.shape[0]:,}  |  Kolona: {df_raw.shape[1]}")
print(df_raw.head(3).T)

df_raw.info(memory_usage="deep")


1. MBLEDHJA E TË DHËNAVE


Rreshta: 46,831  |  Kolona: 42
                                                                               0                                                  1                                                  2
gbifID                                                                6500474327                                         6500462703                                         6500450749
scientificName                                    Mantis religiosa (Linne, 1758)                  Issoria lathonia (Linnaeus, 1758)                  Podarcis muralis (Laurenti, 1768)
species                                                         Mantis religiosa                                   Issoria lathonia                                   Podarcis muralis
vernacularName                                                    Praying Mantis                          Queen of Spain Fritillary                                 Common Wall Lizard
kingdom                                               

## Madhësia e skedarëve

In [3]:
section("Madhësia e skedarëve")
GITHUB_LIMIT_MB = 100


def file_size_mb(path):
    return path.stat().st_size / 1024**2


print(f"Dataseti origjinal: {file_size_mb(DATA_PATH):.1f} MB")
print(f"Në memorie (pandas): {df_raw.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

drop_planned = ["iasAppliesFrom", "establishmentMeans", "identifiedBy", "iucnRedListStatus",
                "taxonKey", "catalogNumber", "collectionCode", "issue"]
cleaned_mb = len(df_raw.drop(columns=drop_planned).to_csv(index=False).encode("utf-8")) / 1024**2
sample_mb = len(df_raw.sample(frac=0.1, random_state=42).to_csv(index=False).encode("utf-8")) / 1024**2
print(f"Versioni i pastruar (vlerësim): {cleaned_mb:.1f} MB")
print(f"Mostra 10% (vlerësim): {sample_mb:.1f} MB")
print(f"Kufiri i GitHub: {GITHUB_LIMIT_MB} MB")


Madhësia e skedarëve
Dataseti origjinal: 22.0 MB
Në memorie (pandas): 31.9 MB


Versioni i pastruar (vlerësim): 17.5 MB
Mostra 10% (vlerësim): 2.3 MB
Kufiri i GitHub: 100 MB


## 2. DEFINIMI I TIPEVE TË TË DHËNAVE

In [4]:
section("2. DEFINIMI I TIPEVE TË TË DHËNAVE")

# Për secilën kolonë: tipi konceptual dhe tipi i synuar në pandas (shih reports/tipet_e_te_dhenave.md)
TYPES = pd.DataFrame([
    ("gbifID",                        "ID",       "int64"),
    ("scientificName",                "nominal",  "category"),
    ("species",                       "nominal",  "category"),
    ("vernacularName",                "nominal",  "category"),
    ("kingdom",                       "nominal",  "category"),
    ("phylum",                        "nominal",  "category"),
    ("class",                         "nominal",  "category"),
    ("order",                         "nominal",  "category"),
    ("family",                        "nominal",  "category"),
    ("genus",                         "nominal",  "category"),
    ("taxonRank",                     "ordinal",  "category (ordered)"),
    ("taxonKey",                      "ID",       "Int64"),
    ("speciesKey",                    "ID",       "Int64"),
    ("iucnRedListCategory",           "ordinal",  "category (ordered)"),
    ("iucnRedListStatus",             "ordinal",  "category (ordered)"),
    ("eventDate",                     "datë",     "datetime64"),
    ("year",                          "interval", "Int64"),
    ("month",                         "ordinal",  "Int64"),
    ("day",                           "ordinal",  "Int64"),
    ("decimalLatitude",               "interval", "float64"),
    ("decimalLongitude",              "interval", "float64"),
    ("coordinateUncertaintyInMeters", "ratio",    "float64"),
    ("elevation",                     "interval", "float64"),
    ("locality",                      "tekst",    "str"),
    ("municipality",                  "nominal",  "category"),
    ("district",                      "nominal",  "category"),
    ("protectedArea",                 "nominal",  "category"),
    ("protectedAreaDesignation",      "nominal",  "category"),
    ("strictlyProtected",             "binar",    "bool"),
    ("basisOfRecord",                 "nominal",  "category"),
    ("individualCount",               "ratio",    "Int64"),
    ("establishmentMeans",            "nominal",  "category"),
    ("recordedBy",                    "tekst",    "str"),
    ("identifiedBy",                  "tekst",    "str"),
    ("institutionCode",               "nominal",  "category"),
    ("collectionCode",                "nominal",  "category"),
    ("catalogNumber",                 "ID",       "str"),
    ("datasetKey",                    "ID",       "category"),
    ("license",                       "nominal",  "category"),
    ("issue",                         "tekst",    "str"),
    ("iasUnionConcern",               "binar",    "bool"),
    ("iasAppliesFrom",                "datë",     "datetime64"),
], columns=["kolona", "tipi_konceptual", "tipi_i_synuar"]).set_index("kolona")
assert list(TYPES.index) == list(df_raw.columns)

# Kolonat ordinale: renditja e kategorive dhe vlerat që nënkuptojnë "e panjohur" (-> NaN)
ORDERED = {
    "taxonRank": (["KINGDOM", "PHYLUM", "CLASS", "ORDER", "FAMILY", "GENUS",
                   "SPECIES", "SUBSPECIES", "VARIETY", "FORM"], ["UNRANKED"]),
    "iucnRedListCategory": (["LC", "NT", "VU", "EN", "CR", "EW", "EX"], ["NE", "DD"]),
    "iucnRedListStatus": (["Least Concern", "Near Threatened", "Vulnerable", "Endangered",
                           "Critically Endangered", "Extinct in the Wild", "Extinct"],
                          ["Not Evaluated", "Data Deficient"]),
}

df = df_raw.copy()
for col, target in TYPES.tipi_i_synuar.items():
    if col in ORDERED:
        cats, unknown = ORDERED[col]
        df[col] = pd.Categorical(df[col].replace({u: np.nan for u in unknown}),
                                 categories=cats, ordered=True)
    elif target == "datetime64":
        # te intervalet "fillimi/fundi" merret data e fillimit; ora hiqet
        start = df[col].str.split("/").str[0].str[:10]
        df[col] = pd.to_datetime(start, format="ISO8601", errors="coerce")
    elif target == "str":
        pass  # teksti mbetet ashtu siç lexohet (str në pandas 3, object në pandas 2)
    elif target != str(df[col].dtype):
        df[col] = df[col].astype(target)


def dtype_label(s):
    if isinstance(s.dtype, pd.CategoricalDtype):
        return "category (ordered)" if s.cat.ordered else "category"
    if s.dtype.kind == "M":
        return "datetime64"
    return "str" if s.dtype.kind in "OUT" or isinstance(s.dtype, pd.StringDtype) else str(s.dtype)


type_report = TYPES.assign(
    tipi_pandas=df_raw.dtypes.astype(str),
    tipi_pas=[dtype_label(df[c]) for c in TYPES.index],
    vlera_unike=df_raw.nunique(),
    na_para=df_raw.isna().sum(),
    na_pas=df.isna().sum(),
)[["tipi_pandas", "tipi_konceptual", "tipi_i_synuar", "tipi_pas", "vlera_unike", "na_para", "na_pas"]]
assert (type_report.tipi_i_synuar == type_report.tipi_pas).all()

print(type_report.to_string())
print("\nKolona sipas tipit konceptual:")
print(type_report.tipi_konceptual.value_counts().to_string())

mem_before = df_raw.memory_usage(deep=True).sum() / 1024**2
mem_after = df.memory_usage(deep=True).sum() / 1024**2
print(f"\nMemoria: {mem_before:.1f} MB -> {mem_after:.1f} MB")

# Vlerat që u bënë NaN gjatë konvertimit
new_na = type_report[type_report.na_pas != type_report.na_para]
print("\nVlera që u kthyen në NaN gjatë konvertimit:")
print((new_na.na_pas - new_na.na_para).rename("NaN_te_reja").to_string())


2. DEFINIMI I TIPEVE TË TË DHËNAVE
                              tipi_pandas tipi_konceptual       tipi_i_synuar            tipi_pas  vlera_unike  na_para  na_pas
kolona                                                                                                                         
gbifID                              int64              ID               int64               int64        46831        0       0
scientificName                        str         nominal            category            category         4647       37      37
species                               str         nominal            category            category         4095     1213    1213
vernacularName                        str         nominal            category            category         3314     3873    3873
kingdom                               str         nominal            category            category            7       37      37
phylum                                str         nominal           

## 3.1 VLERAT E ZBRAZËTA

In [5]:
section("3.1 VLERAT E ZBRAZËTA")
missing = pd.DataFrame({
    "mungojne": df_raw.isna().sum(),
    "perqindja": (df_raw.isna().mean() * 100).round(2),
}).sort_values("perqindja", ascending=False)
print(missing[missing.mungojne > 0])

print("\nRreshta pa asnjë vlerë që mungon:", df_raw.dropna().shape[0])
print("Kolona komplet:", missing.index[missing.mungojne == 0].tolist())

print(df_raw["iucnRedListCategory"].value_counts(dropna=False))


3.1 VLERAT E ZBRAZËTA


                               mungojne  perqindja
iasAppliesFrom                    46807      99.95
establishmentMeans                46591      99.49
identifiedBy                      39096      83.48
elevation                         37011      79.03
coordinateUncertaintyInMeters     32004      68.34
protectedArea                     25762      55.01
protectedAreaDesignation          25762      55.01
individualCount                   17697      37.79
locality                          15251      32.57
collectionCode                    13469      28.76
catalogNumber                     12559      26.82
vernacularName                     3873       8.27
recordedBy                         3853       8.23
institutionCode                    3711       7.92
day                                3219       6.87
month                              2867       6.12
year                               2268       4.84
eventDate                          2190       4.68
issue                          

## 3.2 DUPLIKATET

In [6]:
section("3.2 DUPLIKATET")
print("Rreshta identikë:", df_raw.duplicated().sum())
print("gbifID të përsëritura:", df_raw["gbifID"].duplicated().sum())

key = ["scientificName", "decimalLatitude", "decimalLongitude", "eventDate"]
print("Vëzhgime të mundshme të dyfishta (takson + koordinata + data):", df_raw.duplicated(subset=key).sum())


3.2 DUPLIKATET
Rreshta identikë: 0
gbifID të përsëritura: 0
Vëzhgime të mundshme të dyfishta (takson + koordinata + data): 3336


## 3.3 FORMATET E eventDate

In [7]:
section("3.3 FORMATET E eventDate")
dates = df_raw["eventDate"].dropna()
fmt = (dates.str.replace(r"\d", "9", regex=True)
            .value_counts()
            .rename_axis("formati")
            .to_frame("regjistrime"))
print(fmt)

start = df_raw["eventDate"].str.split("/").str[0].str[:10]
# eventDate është konvertuar te seksioni 2 (data e fillimit)
df["eventDatePrecision"] = pd.Categorical(
    start.str.len().map({4: "year", 7: "month", 10: "day"})
)
df["isDateRange"] = df_raw["eventDate"].str.contains("/", na=False)

print("\nData të palexueshme:", (df["eventDate"].isna() & df_raw["eventDate"].notna()).sum())
print("Intervale datash:", df["isDateRange"].sum())
print(df["eventDatePrecision"].value_counts(dropna=False))


both = df["eventDate"].notna() & df["year"].notna()
mismatch = both & (df["eventDate"].dt.year != df["year"])
print("\nMospërputhje eventDate vs year:", mismatch.sum())
print("Data në të ardhmen:", (df["eventDate"] > pd.Timestamp.today()).sum())


3.3 FORMATET E eventDate
                                     regjistrime
formati                                         
9999-99-99                                 37945
9999-99-99T99:99:99                         2554
9999-99-99T99:99                            2417
9999-99-99T99:99:99.999Z                     595
9999                                         409
9999-99-99/9999-99-99                        398
9999-99                                      173
9999-99-99T99:99:99Z                          67
9999/9999                                     46
9999-99-99T99:99Z                             34
9999-99-99T99:99Z/9999-99-99T99:99Z            3

Data të palexueshme: 0
Intervale datash: 447
eventDatePrecision
day      44013
NaN       2190
year       455
month      173
Name: count, dtype: int64

Mospërputhje eventDate vs year: 0
Data në të ardhmen: 0


## 3.4 VLERAT E DYSHIMTA

In [8]:
section("3.4 VLERAT E DYSHIMTA")
print(df[["year", "decimalLatitude", "decimalLongitude", "coordinateUncertaintyInMeters",
          "elevation", "individualCount"]].describe().T)

LAT, LON = (41.85, 43.27), (20.01, 21.80)
ELEV = (297, 2656)

checks = {
    "Koordinata jashtë Kosovës": ~df.decimalLatitude.between(*LAT) | ~df.decimalLongitude.between(*LON),
    "Lartësia = 0 m": df.elevation == 0,
    "Lartësia jashtë 297–2656 m (pa 0)": df.elevation.notna() & (df.elevation != 0) & ~df.elevation.between(*ELEV),
    "Pasiguria e koordinatave > 10 km": df.coordinateUncertaintyInMeters > 10_000,
    "Viti para 1900": df.year < 1900,
    "Identifikim mbi nivelin e species": df.species.isna(),
}
quality = pd.DataFrame({
    "regjistrime": {k: int(v.sum()) for k, v in checks.items()},
})
quality["perqindja"] = (quality.regjistrime / len(df) * 100).round(2)
print()
print(quality)


ic = df["individualCount"].dropna()
q1, q3 = ic.quantile([0.25, 0.75])
upper = q3 + 1.5 * (q3 - q1)
print(f"\nQ1={q1}, Q3={q3}, kufiri i sipërm IQR={upper}")
print("Outliers:", (ic > upper).sum(), "| Maksimumi:", ic.max())
print(ic.value_counts().head(10))


3.4 VLERAT E DYSHIMTA
                                 count         mean           std        min        25%        50%        75%        max
year                           44563.0  2015.648161     14.666617     1826.0     2018.0     2020.0     2022.0     2026.0
decimalLatitude                46831.0    42.564153      0.214627  41.858976  42.561417  42.579243  42.659067   43.24167
decimalLongitude               46831.0    20.972567      0.345911  20.020786  20.815317  21.048304  21.144554  21.762903
coordinateUncertaintyInMeters  14827.0  4865.172658  37074.786607        1.0      250.0     7071.0     7071.0  4226661.0
elevation                       9820.0   582.211845    634.895781        0.0        0.0      490.0      802.0     3050.0
individualCount                29134.0    22.948273    322.478196        1.0        1.0        2.0        6.0    10200.0

                                   regjistrime  perqindja
Koordinata jashtë Kosovës                    0       0.00
Lartësia = 0 

Outliers: 3630 | Maksimumi: 10200
individualCount
1     11192
2      4852
3      2447
4      1722
5      1294
6       897
7       661
10      604
8       594
9       431
Name: count, dtype: Int64


## 3.5 FLAMUJT E GBIF (issue)

Kolona `issue` ndahet sipas `;`, numërohet çdo flamur dhe % e rreshtave. Çdo flamur klasifikohet si **serioz** (vlera e një kolone që përdoret në analizë mungon, u zëvendësua ose është e pasigurt) ose **informativ** (GBIF plotësoi ose standardizoi një vlerë, ose flamuri prek një fushë që nuk përdoret). Përshkrimi është te `reports/flamujt_gbif.md`.

In [9]:
section("3.5 FLAMUJT E GBIF (issue)")
# Klasifikimi i çdo flamuri: (rëndësia, kolona që prek, arsyeja).
# Serioz = vlera e një kolone që përdoret në analizë mungon, u zëvendësua ose është e pasigurt.
# Informativ = GBIF plotësoi/standardizoi një vlerë, ose flamuri prek një fushë që nuk përdoret.
# Përshkrimet: https://gbif.github.io/gbif-api/apidocs/org/gbif/api/vocabulary/OccurrenceIssue.html
REGISTRY = "Përputhja me regjistrin GRSciColl; prek vetëm metadatat e institucionit, jo vëzhgimin."
GBIF_FLAGS = {
    # --- seriozë ---
    "TAXON_MATCH_HIGHERRANK": ("serioz", "species", "Taksoni u përputh vetëm në rang më të lartë; identiteti i species humbet."),
    "TAXON_MATCH_FUZZY": ("serioz", "scientificName", "Emri u përputh në mënyrë të përafërt; taksoni mund të jetë i gabuar."),
    "TAXON_MATCH_NONE": ("serioz", "kingdom", "Asnjë përputhje me backbone-in; e gjithë taksonomia mungon."),
    "TAXON_MATCH_NAME_AND_ID_AMBIGUOUS": ("serioz", "scientificName", "ID-ja dhe emri çojnë te taksone të ndryshme."),
    "SCIENTIFIC_NAME_AND_ID_INCONSISTENT": ("serioz", "scientificName", "Emri nuk përputhet me emrin e ID-së së regjistruar."),
    "RECORDED_DATE_INVALID": ("serioz", "eventDate", "Datë e pavlefshme te burimi; GBIF nuk e interpreton dot."),
    "RECORDED_DATE_MISMATCH": ("serioz", "eventDate", "eventDate kundërshton year/month/day te burimi; data është e pasigurt."),
    "INDIVIDUAL_COUNT_INVALID": ("serioz", "individualCount", "Numri nuk është numër i plotë pozitiv dhe hiqet."),
    "BASIS_OF_RECORD_INVALID": ("serioz", "basisOfRecord", "Vlera nuk lexohet dhe zëvendësohet me OCCURRENCE."),
    "COORDINATE_UNCERTAINTY_METERS_INVALID": ("serioz", "coordinateUncertaintyInMeters", "Pasiguria është e pavlefshme dhe hiqet."),
    # --- informativë ---
    "CONTINENT_DERIVED_FROM_COORDINATES": ("informativ", "continent (jo në dataset)", "Kontinenti u nxor nga koordinatat, që janë të vlefshme."),
    "COUNTRY_DERIVED_FROM_COORDINATES": ("informativ", "countryCode (jo në dataset)", "Vendi u nxor nga koordinatat; të gjitha pikat janë brenda Kosovës."),
    "COUNTRY_INVALID": ("informativ", "countryCode (jo në dataset)", "Vlera e vendit nuk u lexua; vendi u nxor nga koordinatat brenda Kosovës."),
    "COORDINATE_ROUNDED": ("informativ", "decimalLatitude, decimalLongitude", "Koordinata u rrumbullakua në 5 shifra dhjetore (~1 m)."),
    "GEODETIC_DATUM_ASSUMED_WGS84": ("informativ", "decimalLatitude, decimalLongitude", "Datumi mungonte dhe u supozua WGS84; zhvendosja e mundshme është më e vogël se pasiguria e zakonshme."),
    "COORDINATE_PRECISION_INVALID": ("informativ", "coordinatePrecision (jo në dataset)", "Fusha nuk është në dataset; koordinatat janë të vlefshme."),
    "TAXON_CONCEPT_ID_NOT_FOUND": ("informativ", "taxonKey", "ID-ja e burimit nuk u gjet, por taksoni u përputh me emrin."),
    "TAXON_ID_NOT_FOUND": ("informativ", "taxonKey", "ID-ja e burimit nuk u gjet, por taksoni u përputh me emrin."),
    "SCIENTIFIC_NAME_ID_NOT_FOUND": ("informativ", "taxonKey", "ID-ja e emrit nuk u gjet, por taksoni u përputh me emrin."),
    "OCCURRENCE_STATUS_INFERRED_FROM_INDIVIDUAL_COUNT": ("informativ", "occurrenceStatus (jo në dataset)", "Statusi u nxor nga individualCount."),
    "INSTITUTION_MATCH_NONE": ("informativ", "institutionCode", REGISTRY),
    "INSTITUTION_MATCH_FUZZY": ("informativ", "institutionCode", REGISTRY),
    "AMBIGUOUS_INSTITUTION": ("informativ", "institutionCode", REGISTRY),
    "COLLECTION_MATCH_NONE": ("informativ", "collectionCode", REGISTRY),
    "COLLECTION_MATCH_FUZZY": ("informativ", "collectionCode", REGISTRY),
    "AMBIGUOUS_COLLECTION": ("informativ", "collectionCode", REGISTRY),
    "INSTITUTION_COLLECTION_MISMATCH": ("informativ", "institutionCode", REGISTRY),
    "REFERENCES_URI_INVALID": ("informativ", "references (jo në dataset)", "Link i pavlefshëm; fusha nuk përdoret."),
    "MULTIMEDIA_URI_INVALID": ("informativ", "media (jo në dataset)", "Link i pavlefshëm; fusha nuk përdoret."),
    "NUCLEOTIDE_SEQUENCE_GAPS_REMOVED": ("informativ", "sekuenca e ADN-së (jo në dataset)", "Përpunim i sekuencës së ADN-së; nuk përdoret."),
    "NUCLEOTIDE_SEQUENCE_ENDS_TRIMMED": ("informativ", "sekuenca e ADN-së (jo në dataset)", "Përpunim i sekuencës së ADN-së; nuk përdoret."),
    "TYPE_STATUS_INVALID": ("informativ", "typeStatus (jo në dataset)", "Statusi i ekzemplarit tip; nuk përdoret."),
    "SUSPECTED_TYPE": ("informativ", "typeStatus (jo në dataset)", "Statusi i ekzemplarit tip me pasiguri; nuk përdoret."),
    "AGE_OR_STAGE_RANK_MISMATCH": ("informativ", "konteksti gjeologjik (jo në dataset)", "Mosha gjeologjike e fosileve; nuk përdoret."),
    "IDENTIFIED_DATE_INVALID": ("informativ", "dateIdentified (jo në dataset)", "Data e identifikimit; data e vëzhgimit nuk preket."),
}

flag_lists = df_raw["issue"].dropna().str.split(";")
counts = flag_lists.explode().value_counts()
unknown = sorted(set(counts.index) - set(GBIF_FLAGS))
assert not unknown, f"Flamuj pa klasifikim: {unknown}"

flag_sets = df_raw["issue"].fillna("").str.split(";").apply(set)


def empty_in_column(flag, col):
    """Te flamujt seriozë: sa nga rreshtat me këtë flamur e kanë kolonën përkatëse bosh."""
    if GBIF_FLAGS[flag][0] != "serioz" or col not in df_raw.columns:
        return np.nan
    return int(df_raw.loc[flag_sets.apply(lambda s: flag in s), col].isna().sum())


issues = pd.DataFrame([
    {"Flamuri": f, "n": int(n), "%": round(n / len(df_raw) * 100, 2), "Rëndësia": GBIF_FLAGS[f][0],
     "Kolona": GBIF_FLAGS[f][1], "Bosh te kolona": empty_in_column(f, GBIF_FLAGS[f][1]),
     "Arsyeja": GBIF_FLAGS[f][2]}
    for f, n in counts.items()
])
issues["Bosh te kolona"] = issues["Bosh te kolona"].astype("Int64")
issues = (issues.sort_values(["Rëndësia", "n"], ascending=[False, False])
          .set_index("Flamuri"))

print(f"Flamuj të ndryshëm: {len(issues)}  |  Rreshta pa asnjë flamur: {int(df_raw.issue.isna().sum()):,}")
print("\nTabela: Flamuri | n | % | Rëndësia")
display(issues.style.set_properties(subset=["Arsyeja", "Kolona"], **{"text-align": "left"})
        .format({"n": "{:,}", "%": "{:.2f}"}, na_rep="–"))
print("% = rreshta me flamurin / të gjithë rreshtat (46,831). Një rresht mund të ketë disa flamuj.")
print("'Bosh te kolona' (vetëm te seriozët) = sa nga rreshtat me flamurin e kanë kolonën NaN në dataset.")

SERIOUS_FLAGS = [f for f, (r, _, _) in GBIF_FLAGS.items() if r == "serioz"]
has_serious = flag_sets.apply(lambda s: bool(s & set(SERIOUS_FLAGS)))
has_any = df_raw.issue.notna()
flag_rows = pd.Series({
    "pa asnjë flamur": int((~has_any).sum()),
    "vetëm flamuj informativë": int((has_any & ~has_serious).sum()),
    "të paktën një flamur serioz": int(has_serious.sum()),
}, name="rreshta").to_frame()
flag_rows["perqindja"] = (flag_rows.rreshta / len(df_raw) * 100).round(2)
flag_rows.index.name = "rreshti ka"
print("\nRreshtat sipas flamujve")
display(flag_rows)
by_level = issues.groupby("Rëndësia").agg(flamuj=("n", "size"), paraqitje=("n", "sum"))
display(by_level)


3.5 FLAMUJT E GBIF (issue)


Flamuj të ndryshëm: 35  |  Rreshta pa asnjë flamur: 1,426

Tabela: Flamuri | n | % | Rëndësia


,n,%,Rëndësia,Kolona,Bosh te kolona,Arsyeja
Flamuri,,,,,,
TAXON_MATCH_HIGHERRANK,756,1.61,serioz,species,702,Taksoni u përputh vetëm në rang më të lartë; identiteti i species humbet.
RECORDED_DATE_INVALID,508,1.08,serioz,eventDate,497,Datë e pavlefshme te burimi; GBIF nuk e interpreton dot.
INDIVIDUAL_COUNT_INVALID,351,0.75,serioz,individualCount,351,Numri nuk është numër i plotë pozitiv dhe hiqet.
TAXON_MATCH_FUZZY,190,0.41,serioz,scientificName,0,Emri u përputh në mënyrë të përafërt; taksoni mund të jetë i gabuar.
BASIS_OF_RECORD_INVALID,120,0.26,serioz,basisOfRecord,0,Vlera nuk lexohet dhe zëvendësohet me OCCURRENCE.
RECORDED_DATE_MISMATCH,87,0.19,serioz,eventDate,2,eventDate kundërshton year/month/day te burimi; data është e pasigurt.
TAXON_MATCH_NONE,28,0.06,serioz,kingdom,28,Asnjë përputhje me backbone-in; e gjithë taksonomia mungon.
COORDINATE_UNCERTAINTY_METERS_INVALID,7,0.01,serioz,coordinateUncertaintyInMeters,7,Pasiguria është e pavlefshme dhe hiqet.
TAXON_MATCH_NAME_AND_ID_AMBIGUOUS,1,0.00,serioz,scientificName,0,ID-ja dhe emri çojnë te taksone të ndryshme.


% = rreshta me flamurin / të gjithë rreshtat (46,831). Një rresht mund të ketë disa flamuj.
'Bosh te kolona' (vetëm te seriozët) = sa nga rreshtat me flamurin e kanë kolonën NaN në dataset.

Rreshtat sipas flamujve


,rreshta,perqindja
rreshti ka,,
pa asnjë flamur,1426,3.04
vetëm flamuj informativë,43760,93.44
të paktën një flamur serioz,1645,3.51


,flamuj,paraqitje
Rëndësia,,
informativ,25,108603
serioz,10,2049


## 3.6 KONSISTENCA E VLERAVE KATEGORIKE

In [10]:
section("3.6 KONSISTENCA E VLERAVE KATEGORIKE")
for c in ["kingdom", "taxonRank", "basisOfRecord", "district", "license", "strictlyProtected", "iasUnionConcern"]:
    print(f"--- {c} ---")
    print(df[c].value_counts(dropna=False).to_string(), "\n")


3.6 KONSISTENCA E VLERAVE KATEGORIKE
--- kingdom ---
kingdom
Animalia          42003
Plantae            3740
Fungi              1006
Chromista            38
NaN                  37
Protozoa              4
incertae sedis        2
Bacteria              1 

--- taxonRank ---
taxonRank
SPECIES       44727
GENUS           925
SUBSPECIES      579
NaN             251
FORM            132
VARIETY          60
FAMILY           58
ORDER            41
KINGDOM          27
PHYLUM           17
CLASS            14 

--- basisOfRecord ---
basisOfRecord
HUMAN_OBSERVATION      42479
PRESERVED_SPECIMEN      1748
OCCURRENCE              1269
MATERIAL_SAMPLE          641
MACHINE_OBSERVATION      376
MATERIAL_CITATION        317
OBSERVATION                1 

--- district ---
district
Prishtinë / Priština              25734
Prizren                            5268
Gjilan / Gnjilane                  4156
Mitrovicë / Kosovska Mitrovica     3196
Pejë / Peć                         3187
Gjakovë / Đakovica         

## 3.7 KORNIZA E KUALITETIT

Pesë dimensione: plotësia, vlefshmëria, konsistenca, unikaliteti dhe aktualiteti. Për secilin kontroll: metrika, rregulli dhe kolonat. Përshkrimi i plotë është te `reports/korniza_e_kualitetit.md`.

In [11]:
section("3.7 KORNIZA E KUALITETIT")
# Çdo kontroll ka: dimensionin, kolonat, rreshtat ku aplikohet dhe rreshtat që nuk e plotësojnë.
# Metrika = rreshta që e plotësojnë / rreshta ku aplikohet (shih reports/korniza_e_kualitetit.md).

TODAY = pd.Timestamp.today().normalize()
KEY_COLS = ["species", "eventDate", "decimalLatitude", "decimalLongitude",
            "municipality", "district", "basisOfRecord", "kingdom"]

VOCAB = {
    "kingdom": {"Animalia", "Plantae", "Fungi", "Chromista", "Protozoa", "Bacteria",
                "Archaea", "Viruses", "incertae sedis"},
    "taxonRank": {"KINGDOM", "PHYLUM", "CLASS", "ORDER", "FAMILY", "GENUS", "SPECIES",
                  "SUBSPECIES", "VARIETY", "FORM", "UNRANKED"},
    "basisOfRecord": {"HUMAN_OBSERVATION", "MACHINE_OBSERVATION", "OBSERVATION", "PRESERVED_SPECIMEN",
                      "FOSSIL_SPECIMEN", "LIVING_SPECIMEN", "MATERIAL_SAMPLE", "MATERIAL_CITATION",
                      "OCCURRENCE"},
    "iucnRedListCategory": {"LC", "NT", "VU", "EN", "CR", "EW", "EX", "NE", "DD"},
    "license": {"CC0_1_0", "CC_BY_4_0", "CC_BY_NC_4_0"},
    "establishmentMeans": {"native", "nativeReintroduced", "introduced",
                           "introducedAssistedColonisation", "vagrant", "uncertain"},
}
# Flamujt seriozë të GBIF, nga klasifikimi te seksioni 3.5
GBIF_INVALID = SERIOUS_FLAGS

ALL = pd.Series(True, index=df.index)
rules = []


def add_rule(id_, dim, kontrolli, kolonat, njesia, n, n_fail, veprimi):
    rules.append({"id": id_, "dimensioni": dim, "kontrolli": kontrolli, "kolonat": kolonat,
                  "njesia": njesia, "aplikohen": n, "nuk_plotesojne": n_fail,
                  "perqindja": round((1 - n_fail / n) * 100, 2) if n else np.nan,
                  "veprimi": veprimi})


def rule(id_, dim, kontrolli, kolonat, fail, applies=ALL, veprimi=""):
    fail = pd.Series(fail, index=df.index).fillna(False).astype(bool) & applies
    add_rule(id_, dim, kontrolli, kolonat, "rreshta", int(applies.sum()), int(fail.sum()), veprimi)
    return fail


def many_to_one(child, parent, within=()):
    """Rreshtat ku vlera e `child` (brenda grupit `within`) lidhet me më shumë se një `parent`."""
    keys = [*within, child]
    n_parents = df_raw.groupby(keys)[parent].transform("nunique")
    return (n_parents > 1) & df_raw[keys + [parent]].notna().all(axis=1)


# --- Plotësia ---
D = "Plotësia"
rule("P1", D, "Të gjitha kolonat kryesore të plota", ", ".join(KEY_COLS),
     df[KEY_COLS].isna().any(axis=1), veprimi="analiza kryesore vetëm me rreshtat e plotë")
rule("P2", D, "Identifikim deri në species", "species", df.species.isna(),
     veprimi="përjashto nga analiza sipas species")
rule("P3", D, "Data e njohur", "eventDate", df.eventDate.isna(),
     veprimi="përjashto nga analiza kohore")
rule("P4", D, "Statusi IUCN i vlerësuar (NE/DD = mungesë)", "iucnRedListCategory",
     df.iucnRedListCategory.isna(), veprimi="shëno 'i pavlerësuar'")
rule("P5", D, "Lartësia e njohur (0 m = mungesë)", "elevation",
     df.elevation.isna() | (df.elevation == 0), veprimi="0 -> NaN; përdor vetëm si veti ndihmëse")
rule("P6", D, "Pasiguria e koordinatave e dhënë", "coordinateUncertaintyInMeters",
     df.coordinateUncertaintyInMeters.isna(), veprimi="shëno 'e panjohur' te diskretizimi")
rule("P7", D, "Numri i individëve i dhënë", "individualCount", df.individualCount.isna(),
     veprimi="mos e imputo; analizo vetëm vlerat e dhëna")
sparse = df_raw.columns[df_raw.isna().mean() > 0.8]
add_rule("P8", D, "Kolona me jo më shumë se 80% mungesa", ", ".join(sparse), "kolona",
         df_raw.shape[1], len(sparse), "hiq kolonat")

# --- Vlefshmëria ---
D = "Vlefshmëria"
LAT, LON, ELEV = (41.85, 43.27), (20.01, 21.80), (297, 2656)
has_coord = df.decimalLatitude.notna() & df.decimalLongitude.notna()
rule("V1", D, f"Koordinatat brenda Kosovës (lat {LAT[0]}–{LAT[1]}, lon {LON[0]:.2f}–{LON[1]:.2f})", "decimalLatitude, decimalLongitude",
     ~df.decimalLatitude.between(*LAT) | ~df.decimalLongitude.between(*LON), has_coord, "largo rreshtin")
elev = df.elevation.notna() & (df.elevation != 0)
rule("V2", D, f"Lartësia brenda {ELEV[0]}–{ELEV[1]} m", "elevation",
     ~df.elevation.between(*ELEV), elev, "vendos NaN")
unc = df.coordinateUncertaintyInMeters.notna()
rule("V3", D, "Pasiguria > 0 dhe ≤ 10 km", "coordinateUncertaintyInMeters",
     (df.coordinateUncertaintyInMeters <= 0) | (df.coordinateUncertaintyInMeters > 10_000), unc,
     "shëno; përjashto nga analiza hapësinore e detajuar")
rule("V4", D, "eventDate e lexueshme si ISO 8601", "eventDate",
     df.eventDate.isna(), df_raw.eventDate.notna(), "vendos NaN")
rule("V5", D, "Data jo në të ardhmen", "eventDate, year",
     (df.eventDate > TODAY) | (df.year > TODAY.year), df.eventDate.notna() | df.year.notna(), "vendos NaN")
has_ymd = df[["year", "month", "day"]].notna().all(axis=1)
ymd = pd.to_datetime(df.loc[has_ymd, ["year", "month", "day"]].astype("int64"), errors="coerce")
rule("V6", D, "month 1–12, day 1–31 dhe data year-month-day ekziston", "year, month, day",
     ~df.month.between(1, 12) | ~df.day.between(1, 31) | ymd.reindex(df.index).isna(),
     has_ymd, "vendos NaN")
rule("V7", D, "individualCount ≥ 1", "individualCount", df.individualCount < 1,
     df.individualCount.notna(), "vendos NaN")
vocab_fail = pd.concat([df_raw[c].notna() & ~df_raw[c].isin(v) for c, v in VOCAB.items()], axis=1).any(axis=1)
rule("V8", D, "Vlerat kategorike brenda fjalorit të lejuar (GBIF/DwC/IUCN)", ", ".join(VOCAB),
     vocab_fail, veprimi="korrigjo ose vendos NaN")
flags = df_raw.issue.fillna("").str.split(";")
rule("V9", D, "Pa flamuj seriozë të GBIF (seksioni 3.5)", "issue",
     flags.apply(lambda f: any(x in GBIF_INVALID for x in f)), veprimi="shëno me flamur binar")
ic = df.individualCount.dropna()
q1, q3 = ic.quantile([0.25, 0.75])
rule("V10", D, f"individualCount jo outlier (IQR, > {q3 + 1.5 * (q3 - q1):g})", "individualCount",
     df.individualCount > q3 + 1.5 * (q3 - q1), df.individualCount.notna(), "shëno; log1p te transformimi")

# --- Konsistenca ---
D = "Konsistenca"
start = df_raw.eventDate.str.split("/").str[0].str[:10]
prec = start.str.len()
ev = df.eventDate
date_fail = ((ev.dt.year != df.year)
             | ((prec >= 7) & (ev.dt.month != df.month))
             | ((prec == 10) & (ev.dt.day != df.day)))
rule("K1", D, "eventDate përputhet me year/month/day (sipas precizionit)", "eventDate, year, month, day",
     date_fail, ev.notna() & df.year.notna(), "përdor eventDate; plotëso year/month/day prej saj")
tax_pairs = [("genus", "family"), ("family", "order"), ("order", "class"), ("class", "phylum"),
             ("phylum", "kingdom")]
# brenda të njëjtit kingdom: p.sh. Linaria dhe Prunella janë gjini edhe te zogjtë edhe te bimët
rule("K2", D, "Hierarkia taksonomike: çdo takson ka një prind (brenda kingdom)",
     "genus, family, order, class, phylum, kingdom",
     pd.concat([many_to_one(c, p, within=["kingdom"]) for c, p in tax_pairs], axis=1).any(axis=1),
     df.kingdom.notna(), "rishiko taksonët me më shumë se një prind")
sp_gen = df_raw.species.notna() & df_raw.genus.notna()
rule("K3", D, "species fillon me genus", "species, genus",
     df_raw.species.str.split().str[0] != df_raw.genus, sp_gen, "rishiko")
infra = df_raw.taxonRank.isin(["SPECIES", "SUBSPECIES", "VARIETY", "FORM"])
rule("K4", D, "taxonRank SPECIES ose më i ulët ⇔ species e plotë", "taxonRank, species",
     infra != df_raw.species.notna(), df_raw.taxonRank.notna() & (df_raw.taxonRank != "UNRANKED"), "rishiko")
rule("K5", D, "iucnRedListCategory ↔ iucnRedListStatus 1:1", "iucnRedListCategory, iucnRedListStatus",
     many_to_one("iucnRedListCategory", "iucnRedListStatus") | many_to_one("iucnRedListStatus", "iucnRedListCategory"),
     df_raw.iucnRedListCategory.notna(), "mbaj iucnRedListCategory")
rule("K6", D, "Çdo komunë i përket një rajoni", "municipality, district",
     many_to_one("municipality", "district"), veprimi="korrigjo rajonin")
rule("K7", D, "protectedArea dhe protectedAreaDesignation bashkë; një zonë = një lloj",
     "protectedArea, protectedAreaDesignation",
     (df_raw.protectedArea.isna() != df_raw.protectedAreaDesignation.isna())
     | many_to_one("protectedArea", "protectedAreaDesignation"), veprimi="korrigjo llojin")
rule("K8", D, "iasAppliesFrom e plotë ⇒ iasUnionConcern = TRUE", "iasAppliesFrom, iasUnionConcern",
     ~df_raw.iasUnionConcern, df_raw.iasAppliesFrom.notna(), "rishiko")
nominal = TYPES.index[TYPES.tipi_konceptual == "nominal"]
fmt_fail = pd.Series(False, index=df.index)
for c in nominal:
    s = df_raw[c].dropna().astype(str)
    variants = s.drop_duplicates().groupby(s.drop_duplicates().str.strip().str.lower()).size()
    bad = s[(s != s.str.strip()) | s.str.contains("  ", regex=False)
            | s.str.strip().str.lower().isin(variants[variants > 1].index)]
    fmt_fail |= df_raw[c].isin(bad.unique())
rule("K9", D, "Kategoritë pa hapësira të tepërta dhe pa variante të shkronjave", ", ".join(nominal),
     fmt_fail, veprimi="strip + njësim i shkronjave")

# --- Unikaliteti ---
D = "Unikaliteti"
rule("U1", D, "gbifID unik", "gbifID", df_raw.gbifID.duplicated(keep=False), veprimi="largo dyfishin")
rule("U2", D, "Pa rreshta identikë", "të gjitha", df_raw.duplicated(keep="first"), veprimi="largo dyfishin")
rule("U3", D, "catalogNumber unik brenda dataset-it", "datasetKey, catalogNumber",
     df_raw.duplicated(["datasetKey", "catalogNumber"], keep=False), df_raw.catalogNumber.notna(),
     "rishiko (mund të jenë disa pjesë të një ekzemplari)")
DUP_KEY = ["scientificName", "decimalLatitude", "decimalLongitude", "eventDate"]
rule("U4", D, "Pa vëzhgime të dyfishta (takson + koordinata + data)", ", ".join(DUP_KEY),
     df_raw.duplicated(DUP_KEY, keep="first"), df_raw[DUP_KEY].notna().all(axis=1),
     "shëno; vendos te integrimi")
dup_groups = df_raw[df_raw.duplicated(DUP_KEY, keep=False) & df_raw[DUP_KEY].notna().all(axis=1)]
multi_src = dup_groups.groupby(DUP_KEY).datasetKey.transform("nunique") > 1
rule("U5", D, "Pa dyfishime ndërmjet burimeve të ndryshme (datasetKey)", ", ".join(DUP_KEY) + ", datasetKey",
     df_raw.index.isin(multi_src[multi_src].index) & df_raw.duplicated(DUP_KEY, keep="first"),
     df_raw[DUP_KEY].notna().all(axis=1), "mbaj një rresht për grup")

# --- Aktualiteti ---
D = "Aktualiteti"
RECENT = TODAY.year - 9
has_year = df.year.notna()
rule("A1", D, f"Regjistrim nga 10 vitet e fundit (year ≥ {RECENT})", "year", df.year < RECENT, has_year,
     "krahaso periudhat; mos i përziej pa shënim")
rule("A2", D, "Regjistrim nga viti 2000 e tutje", "year", df.year < 2000, has_year,
     "shëno si historik")
last_year = df.groupby("datasetKey", observed=True).year.max()
inactive = last_year[last_year < RECENT].index
rule("A3", D, f"Dataset-i ka regjistrime pas vitit {RECENT - 1}", "datasetKey, year",
     df.datasetKey.isin(inactive), has_year, "shëno burimin si joaktiv")

METRIKA = {
    "P1": "% e rreshtave me të 8 kolonat kryesore të plota",
    "P2": "% e rreshtave me species të plotë",
    "P3": "% e rreshtave me eventDate të plotë",
    "P4": "% e rreshtave me kategori IUCN të vlerësuar (LC–EX)",
    "P5": "% e rreshtave me elevation të plotë dhe ≠ 0",
    "P6": "% e rreshtave me coordinateUncertaintyInMeters të plotë",
    "P7": "% e rreshtave me individualCount të plotë",
    "P8": "% e kolonave me ≤ 80% mungesa",
    "V1": "% e koordinatave brenda kutisë gjeografike të Kosovës",
    "V2": "% e lartësive (≠ 0) brenda 297–2656 m",
    "V3": "% e pasigurive të dhëna në intervalin (0, 10 000] m",
    "V4": "% e eventDate të dhëna që lexohen si datë",
    "V5": "% e datave ≤ data e sotme",
    "V6": "% e treshave year-month-day që formojnë datë reale",
    "V7": "% e individualCount të dhëna ≥ 1",
    "V8": "% e rreshtave me të gjitha vlerat kategorike brenda fjalorit",
    "V9": "% e rreshtave pa asnjë flamur serioz të GBIF",
    "V10": "% e individualCount të dhëna ≤ Q3 + 1.5·IQR",
    "K1": "% e rreshtave ku viti/muaji/dita e eventDate = year/month/day",
    "K2": "% e rreshtave ku çdo takson ka një prind brenda kingdom",
    "K3": "% e rreshtave ku fjala e parë e species = genus",
    "K4": "% e rreshtave ku taxonRank dhe species përputhen",
    "K5": "% e rreshtave ku kodi dhe emri IUCN lidhen 1:1",
    "K6": "% e rreshtave ku komuna lidhet me një rajon",
    "K7": "% e rreshtave ku zona e mbrojtur dhe lloji i saj përputhen",
    "K8": "% e rreshtave me iasAppliesFrom ku iasUnionConcern = TRUE",
    "K9": "% e rreshtave pa hapësira të tepërta ose variante shkronjash",
    "U1": "% e gbifID unike",
    "U2": "% e rreshtave që nuk janë kopje e një rreshti tjetër",
    "U3": "% e catalogNumber unike brenda datasetKey",
    "U4": "% e vëzhgimeve pa dyfish (takson + koordinata + data)",
    "U5": "% e vëzhgimeve pa dyfish nga një dataset tjetër",
    "A1": "% e regjistrimeve me year ≥ 2017",
    "A2": "% e regjistrimeve me year ≥ 2000",
    "A3": "% e regjistrimeve nga dataset-e me të dhëna pas 2016",
}
framework = pd.DataFrame(rules).set_index("id")
framework.insert(1, "metrika", framework.index.map(METRIKA))
assert framework.metrika.notna().all()

print("Tabela: Dimensioni | Metrika | Kontrolli | Kolonat")
display(framework[["dimensioni", "metrika", "kontrolli", "kolonat"]]
        .rename(columns=str.capitalize).style.set_properties(**{"text-align": "left"}))
print()
print("Vlerat fillestare:")
print(framework[["dimensioni", "kontrolli", "njesia", "aplikohen", "nuk_plotesojne", "perqindja"]].to_string())
print("\nKontrolle sipas dimensionit (gjithsej / 100% të plotësuara):")
print(framework.groupby("dimensioni", sort=False).perqindja
      .agg(kontrolle="size", plot=lambda x: int((x == 100).sum())).to_string())
print(f"\nViti median: {int(df.year.median())}  |  Regjistrimi më i ri: {df.eventDate.max().date()}  |  "
      f"Dataset-e joaktive: {len(inactive)} / {df.datasetKey.nunique()}")


3.7 KORNIZA E KUALITETIT


Tabela: Dimensioni | Metrika | Kontrolli | Kolonat


,Dimensioni,Metrika,Kontrolli,Kolonat
id,,,,
P1,Plotësia,% e rreshtave me të 8 kolonat kryesore të plota,Të gjitha kolonat kryesore të plota,"species, eventDate, decimalLatitude, decimalLongitude, municipality, district, basisOfRecord, kingdom"
P2,Plotësia,% e rreshtave me species të plotë,Identifikim deri në species,species
P3,Plotësia,% e rreshtave me eventDate të plotë,Data e njohur,eventDate
P4,Plotësia,% e rreshtave me kategori IUCN të vlerësuar (LC–EX),Statusi IUCN i vlerësuar (NE/DD = mungesë),iucnRedListCategory
P5,Plotësia,% e rreshtave me elevation të plotë dhe ≠ 0,Lartësia e njohur (0 m = mungesë),elevation
P6,Plotësia,% e rreshtave me coordinateUncertaintyInMeters të plotë,Pasiguria e koordinatave e dhënë,coordinateUncertaintyInMeters
P7,Plotësia,% e rreshtave me individualCount të plotë,Numri i individëve i dhënë,individualCount
P8,Plotësia,% e kolonave me ≤ 80% mungesa,Kolona me jo më shumë se 80% mungesa,"establishmentMeans, identifiedBy, iasAppliesFrom"
V1,Vlefshmëria,% e koordinatave brenda kutisë gjeografike të Kosovës,"Koordinatat brenda Kosovës (lat 41.85–43.27, lon 20.01–21.80)","decimalLatitude, decimalLongitude"



Vlerat fillestare:
      dimensioni                                                               kontrolli   njesia  aplikohen  nuk_plotesojne  perqindja
id                                                                                                                                     
P1      Plotësia                                     Të gjitha kolonat kryesore të plota  rreshta      46831            3205      93.16
P2      Plotësia                                             Identifikim deri në species  rreshta      46831            1213      97.41
P3      Plotësia                                                           Data e njohur  rreshta      46831            2190      95.32
P4      Plotësia                              Statusi IUCN i vlerësuar (NE/DD = mungesë)  rreshta      46831           16935      63.84
P5      Plotësia                                       Lartësia e njohur (0 m = mungesë)  rreshta      46831           40645      13.21
P6      Plotësia            

## 3.8 PLOTËSIA PËR KOLONË DHE PËR RRESHT

Plotësia për kolonë llogaritet edhe me **mungesat e fshehura**, vlera që duken të plota por nënkuptojnë "e panjohur" (`elevation = 0`, `NE`/`DD`, `UNRANKED`, `unknown`). NaN te `protectedArea` (jashtë zonës) dhe te `iasAppliesFrom` (vlen vetëm për speciet pushtuese) nuk numërohen si mungesa.

Plotësia për rresht: sa rreshta i kanë të plota fushat kyçe **species, data, koordinata dhe komuna**. Përshkrimi është te `reports/plotesia.md`.

In [12]:
section("3.8 PLOTËSIA PËR KOLONË DHE PËR RRESHT")
N = len(df_raw)


def norm(col):
    return df_raw[col].astype("string").str.strip().str.lower()


# Mungesat e fshehura: vlera që duken të plota, por nënkuptojnë "e panjohur"
HIDDEN = {
    "elevation": ("0 m", df_raw.elevation == 0),
    "iucnRedListCategory": ("NE, DD", df_raw.iucnRedListCategory.isin(["NE", "DD"])),
    "iucnRedListStatus": ("Not Evaluated, Data Deficient",
                          df_raw.iucnRedListStatus.isin(["Not Evaluated", "Data Deficient"])),
    "taxonRank": ("UNRANKED", df_raw.taxonRank == "UNRANKED"),
    "locality": ("unknown", norm("locality") == "unknown"),
    "recordedBy": ("unknown", norm("recordedBy") == "unknown"),
    "identifiedBy": ("0, Anon., unknown", norm("identifiedBy").isin(["0", "anon.", "unknown"])),
}
# Mungesat strukturore: NaN ka kuptim ose kolona nuk vlen për rreshtin
NAN_MEANS_VALUE = {
    "protectedArea": "NaN = jashtë zonës së mbrojtur",
    "protectedAreaDesignation": "NaN = jashtë zonës së mbrojtur",
}
APPLIES = {"iasAppliesFrom": ("vlen vetëm kur iasUnionConcern = TRUE", df_raw.iasUnionConcern)}

# Mungesa që mund të plotësohen nga eventDate (sipas precizionit të datës)
start = df_raw.eventDate.str.split("/").str[0].str[:10]
prec = start.str.len()
RECOVERABLE = {
    "year": df_raw.year.isna() & (prec >= 4),
    "month": df_raw.month.isna() & (prec >= 7),
    "day": df_raw.day.isna() & (prec == 10),
}

applies = pd.DataFrame(True, index=df_raw.index, columns=df_raw.columns)
missing_real = df_raw.isna()
for c, (_, mask) in APPLIES.items():
    applies[c] = mask
for c in NAN_MEANS_VALUE:
    missing_real[c] = False
for c, (_, mask) in HIDDEN.items():
    missing_real[c] |= mask.fillna(False)
missing_real &= applies

col_completeness = pd.DataFrame({
    "aplikohen": applies.sum(),
    "nan": df_raw.isna().sum(),
    "te_fshehura": pd.Series({c: int(m.fillna(False).sum()) for c, (_, m) in HIDDEN.items()}),
    "vlerat_e_fshehura": pd.Series({c: v for c, (v, _) in HIDDEN.items()}),
    "mungesa_reale": missing_real.sum(),
    "te_rikuperueshme": pd.Series({c: int(m.sum()) for c, m in RECOVERABLE.items()}),
}).reindex(df_raw.columns)
int_cols = ["aplikohen", "nan", "te_fshehura", "mungesa_reale", "te_rikuperueshme"]
col_completeness[int_cols] = col_completeness[int_cols].fillna(0).astype(int)
col_completeness["vlerat_e_fshehura"] = col_completeness.vlerat_e_fshehura.fillna("")
col_completeness["plotesia_e_dukshme"] = (df_raw.notna().mean() * 100).round(2)
col_completeness["plotesia_reale"] = ((1 - col_completeness.mungesa_reale / col_completeness.aplikohen)
                                      * 100).round(2)
col_completeness["shenim"] = pd.Series({**NAN_MEANS_VALUE, **{c: v for c, (v, _) in APPLIES.items()},
                                        **{c: "plotësohet nga eventDate" for c, m in RECOVERABLE.items()
                                           if m.any()}}).reindex(df_raw.columns).fillna("")
col_completeness = col_completeness.sort_values(["plotesia_reale", "plotesia_e_dukshme"])
col_completeness.index.name = "kolona"

print("Tabela e plotësisë për kolonë (renditur nga më e ulëta)")
display(col_completeness)
diff = col_completeness[col_completeness.plotesia_reale != col_completeness.plotesia_e_dukshme]
print(f"Kolona ku plotësia reale ndryshon nga ajo e dukshme: {len(diff)}")
print(f"Kolona 100% të plota: {(col_completeness.plotesia_reale == 100).sum()} / {df_raw.shape[1]}")

# --- Plotësia për rresht: fushat kyçe ---
KEY_FIELDS = {
    "species": df.species.notna(),
    "data (eventDate)": df.eventDate.notna(),
    "koordinata (lat + lon)": df.decimalLatitude.notna() & df.decimalLongitude.notna(),
    "komuna (municipality)": df.municipality.notna(),
}
key = pd.DataFrame(KEY_FIELDS)
n_missing = (~key).sum(axis=1)


def share(counts, tabela):
    out = counts.rename("rreshta").to_frame()
    out["perqindja"] = (out.rreshta / N * 100).round(2)
    out.insert(0, "tabela", tabela)
    out.index.name = "kategoria"
    return out


per_field = share(key.sum().astype(int), "fusha kyçe e plotë")
all_key = share(pd.Series({"të 4 fushat kyçe të plota": int((n_missing == 0).sum()),
                           "të paktën një fushë kyçe mungon": int((n_missing > 0).sum())}), "rreshta të plotë")
by_count = share(n_missing.value_counts().sort_index().rename(lambda k: "1 fushë kyçe mungon" if k == 1 else f"{k} fusha kyçe mungojnë"),
                 "numri i fushave që mungojnë")
pattern = (~key).apply(lambda r: " + ".join(k.split(" ")[0] for k, v in r.items() if v), axis=1)
by_pattern = share(pattern[n_missing > 0].value_counts().rename(lambda p: f"mungon: {p}"),
                   "çfarë mungon")

row_share = (1 - missing_real.sum(axis=1) / applies.sum(axis=1)) * 100
bins = pd.cut(row_share, [0, 50, 75, 90, 100], right=False, include_lowest=True,
              labels=["< 50%", "50–75%", "75–90%", "90–100%"])
bins = bins.cat.add_categories("100%")
bins[row_share == 100] = "100%"
by_bins = share(bins.value_counts().reindex(["< 50%", "50–75%", "75–90%", "90–100%", "100%"]),
                "plotësia e të 42 kolonave")

row_completeness = pd.concat([all_key, per_field, by_count, by_pattern, by_bins])
print("\nTabela e plotësisë për rresht")
display(row_completeness)
print(f"Plotësia mesatare e një rreshti (42 kolona, me mungesat e fshehura): {row_share.mean():.2f}%  |  "
      f"mediana: {row_share.median():.2f}%")


3.8 PLOTËSIA PËR KOLONË DHE PËR RRESHT
Tabela e plotësisë për kolonë (renditur nga më e ulëta)


,aplikohen,nan,te_fshehura,vlerat_e_fshehura,mungesa_reale,te_rikuperueshme,plotesia_e_dukshme,plotesia_reale,shenim
kolona,,,,,,,,,
establishmentMeans,46831,46591,0,,46591,0,0.51,0.51,
elevation,46831,37011,3634,0 m,40645,0,20.97,13.21,
identifiedBy,46831,39096,15,"0, Anon., unknown",39111,0,16.52,16.48,
coordinateUncertaintyInMeters,46831,32004,0,,32004,0,31.66,31.66,
individualCount,46831,17697,0,,17697,0,62.21,62.21,
iucnRedListCategory,46831,1213,15722,"NE, DD",16935,0,97.41,63.84,
iucnRedListStatus,46831,1213,15722,"Not Evaluated, Data Deficient",16935,0,97.41,63.84,
locality,46831,15251,38,unknown,15289,0,67.43,67.35,
collectionCode,46831,13469,0,,13469,0,71.24,71.24,


Kolona ku plotësia reale ndryshon nga ajo e dukshme: 9
Kolona 100% të plota: 13 / 42



Tabela e plotësisë për rresht


,tabela,rreshta,perqindja
kategoria,,,
të 4 fushat kyçe të plota,rreshta të plotë,43626,93.16
të paktën një fushë kyçe mungon,rreshta të plotë,3205,6.84
species,fusha kyçe e plotë,45618,97.41
data (eventDate),fusha kyçe e plotë,44641,95.32
koordinata (lat + lon),fusha kyçe e plotë,46831,100.00
komuna (municipality),fusha kyçe e plotë,46831,100.00
0 fusha kyçe mungojnë,numri i fushave që mungojnë,43626,93.16
1 fushë kyçe mungon,numri i fushave që mungojnë,3007,6.42
2 fusha kyçe mungojnë,numri i fushave që mungojnë,198,0.42


Plotësia mesatare e një rreshti (42 kolona, me mungesat e fshehura): 85.49%  |  mediana: 90.24%


## 3.9 VLEFSHMËRIA

Kontrollet: koordinatat brenda Kosovës, diapazoni i viteve, `month` 1–12 dhe `day` 1–31, formatet e `eventDate`, frekuenca e vlerës 7071 te `coordinateUncertaintyInMeters` dhe diapazoni i `elevation`. Kolona `korniza` lidh çdo kontroll me ID-në te `reports/korniza_e_kualitetit.md`. Përshkrimi është te `reports/vlefshmeria.md`.

In [13]:
section("3.9 VLEFSHMËRIA")
publisher = df_raw.datasetKey.map(
    pd.read_csv(ROOT / "processed" / "gbif_datasets_cache.csv").set_index("datasetKey").publishingOrg)
validity = []


def check(id_, korniza, kontrolli, kolonat, invalid, applies, veprimi):
    invalid = pd.Series(invalid, index=df.index).fillna(False).astype(bool) & applies
    n = int(applies.sum())
    validity.append({"id": id_, "korniza": korniza, "kontrolli": kontrolli, "kolonat": kolonat,
                     "aplikohen": n, "te_pavlefshme": int(invalid.sum()),
                     "perqindja_e_vlefshme": round((1 - invalid.sum() / n) * 100, 2), "veprimi": veprimi})
    return invalid


# --- 1. Koordinatat brenda Kosovës ---
COORD_FLAGS = ["COUNTRY_COORDINATE_MISMATCH", "ZERO_COORDINATE", "COORDINATE_OUT_OF_RANGE", "COORDINATE_INVALID",
               "PRESUMED_SWAPPED_COORDINATE", "PRESUMED_NEGATED_LATITUDE", "PRESUMED_NEGATED_LONGITUDE"]
flags = df_raw.issue.fillna("").str.split(";")
decimals = df_raw.decimalLatitude.astype(str).str.split(".").str[1].str.len()
check("VL1", "V1", f"Koordinatat brenda kutisë së Kosovës (lat {LAT[0]}–{LAT[1]}, lon {LON[0]:.2f}–{LON[1]:.2f})",
      "decimalLatitude, decimalLongitude",
      ~df.decimalLatitude.between(*LAT) | ~df.decimalLongitude.between(*LON), has_coord, "largo rreshtin")
check("VL2", "V1", "Pika ka komunë të Kosovës", "municipality, decimalLatitude, decimalLongitude",
      df.municipality.isna(), has_coord, "largo rreshtin")
check("VL3", "V1", "Pa flamuj GBIF për koordinata të gabuara", "issue",
      flags.apply(lambda f: any(x in COORD_FLAGS for x in f)), ALL, "largo rreshtin")
check("VL4", "–", "Koordinata me të paktën 3 shifra dhjetore (≤ ~110 m)", "decimalLatitude",
      decimals < 3, has_coord, "shëno; jo për analizë në nivel fshati")

# --- 2. Diapazoni i viteve ---
check("VL5", "V5", f"Viti brenda 1758–{TODAY.year}", "year",
      ~df.year.between(1758, TODAY.year), df.year.notna(), "vendos NaN")
check("VL6", "A2", "Viti ≥ 1900 (jo regjistrim historik)", "year", df.year < 1900, df.year.notna(),
      "shëno si historik")

# --- 3. month 1–12 dhe day 1–31 ---
check("VL7", "V6", "month 1–12", "month", ~df.month.between(1, 12), df.month.notna(), "vendos NaN")
check("VL8", "V6", "day 1–31", "day", ~df.day.between(1, 31), df.day.notna(), "vendos NaN")
check("VL9", "V6", "year-month-day formon datë reale (p.sh. jo 30 shkurt)", "year, month, day",
      ymd.reindex(df.index).isna(), has_ymd, "vendos NaN")

# --- 4. Formatet e eventDate ---
raw_date = df_raw.eventDate
pattern = raw_date.str.replace(r"\d", "9", regex=True)
is_range = raw_date.str.contains("/", na=False)
end = pd.to_datetime(raw_date.where(is_range).str.split("/").str[1].str[:10], format="ISO8601", errors="coerce")
range_days = (end - df.eventDate).dt.days
check("VL10", "V4", "eventDate lexohet si ISO 8601", "eventDate", df.eventDate.isna(), raw_date.notna(),
      "vendos NaN")
check("VL11", "–", "Te intervalet: data e fundit ≥ data e fillimit", "eventDate", range_days < 0, is_range,
      "vendos NaN")
check("VL12", "–", "Te intervalet: gjatësia ≤ 1 vit", "eventDate", range_days > 365, is_range,
      "shëno; jo për analizë sezonale")
check("VL13", "V9", "Pa flamurin GBIF RECORDED_DATE_INVALID", "issue, eventDate",
      flags.apply(lambda f: "RECORDED_DATE_INVALID" in f), ALL, "data mbetet NaN")

# --- 5. Vlera 7071 te coordinateUncertaintyInMeters ---
check("VL14", "V3", "coordinateUncertaintyInMeters > 0 dhe ≤ 10 km", "coordinateUncertaintyInMeters",
      (df.coordinateUncertaintyInMeters <= 0) | (df.coordinateUncertaintyInMeters > 10_000), unc,
      "shëno; jo për analizë hapësinore të detajuar")
GRID_10KM = 7071  # 5000·√2: gjysma e diagonales së një qelize 10 × 10 km
check("VL15", "–", f"coordinateUncertaintyInMeters ≠ {GRID_10KM} (pikë, jo qendër e qelizës 10 km)",
      "coordinateUncertaintyInMeters", df.coordinateUncertaintyInMeters == GRID_10KM, unc,
      "shëno; komuna mund të jetë e pasaktë")

# --- 6. Diapazoni i elevation ---
check("VL16", "P5", "elevation ≠ 0 (0 = e panjohur)", "elevation", df.elevation == 0, df.elevation.notna(),
      "vendos NaN")
check("VL17", "V2", f"elevation brenda {ELEV[0]}–{ELEV[1]} m", "elevation", ~df.elevation.between(*ELEV), elev,
      "vendos NaN")

validity = pd.DataFrame(validity).set_index("id")
validity["perqindja_qe_deshtojne"] = (validity.te_pavlefshme / validity.aplikohen * 100).round(2)

# Tabela e kërkuar: Rregulla | Rreshta që dështojnë | % (e rreshtave ku rregulla aplikohet)
rule_table = pd.DataFrame({
    "Rregulla": validity.index + " – " + validity.kontrolli,
    "Rreshta që dështojnë": validity.te_pavlefshme,
    "%": validity.perqindja_qe_deshtojne,
}).reset_index(drop=True)
print("Tabela: Rregulla | Rreshta që dështojnë | %")
display(rule_table.style.hide(axis="index").set_properties(subset=["Rregulla"], **{"text-align": "left"})
        .format({"Rreshta që dështojnë": "{:,}", "%": "{:.2f}"}))
print("% = rreshta që dështojnë / rreshta ku rregulla aplikohet (ku vlera ekziston) × 100")
print("\nDetajet e kontrolleve")
display(validity)

# --- Tabelat me detaje ---
t_coord = pd.Series({
    "kutia e Kosovës": f"lat {df.decimalLatitude.min():.4f}–{df.decimalLatitude.max():.4f}, "
                       f"lon {df.decimalLongitude.min():.4f}–{df.decimalLongitude.max():.4f}",
    "koordinata të ndryshme": f"{df[['decimalLatitude', 'decimalLongitude']].drop_duplicates().shape[0]:,}",
    "koordinata me 1–2 shifra dhjetore": f"{int((decimals < 3).sum()):,}",
}, name="vlera")
top_points = (df_raw.groupby(["decimalLatitude", "decimalLongitude"])
              .agg(regjistrime=("gbifID", "size"), locality=("locality", "first"),
                   publikuesi=("datasetKey", lambda s: publisher[s.index].mode().iat[0]))
              .sort_values("regjistrime", ascending=False).head(5))
print("\nKoordinatat")
display(t_coord.to_frame())
print("Pikat më të përsëritura (hotspot-e të vëzhgimit, jo qendra të vendit):")
display(top_points)

year_bins = pd.cut(df.year.astype("float"), [1758, 1900, 1950, 2000, 2010, 2020, TODAY.year + 1], right=False,
                   labels=["1758–1899", "1900–1949", "1950–1999", "2000–2009", "2010–2019", f"2020–{TODAY.year}"])
t_years = year_bins.value_counts(sort=False).rename("regjistrime").to_frame()
t_years["perqindja"] = (t_years.regjistrime / df.year.notna().sum() * 100).round(2)
print(f"\nViti: min {int(df.year.min())}, max {int(df.year.max())}, median {int(df.year.median())}")
display(t_years)

t_formats = (pd.DataFrame({"formati": pattern, "interval": is_range, "lexohet": df.eventDate.notna()})
             .dropna(subset=["formati"])
             .groupby("formati")
             .agg(regjistrime=("lexohet", "size"), interval=("interval", "first"), te_lexueshme=("lexohet", "sum"))
             .sort_values("regjistrime", ascending=False))
t_formats["precizioni"] = t_formats.index.str.split("/").str[0].str[:10].str.len().map(
    {4: "vit", 7: "muaj", 10: "ditë"})
t_formats["perqindja"] = (t_formats.regjistrime / raw_date.notna().sum() * 100).round(2)
print(f"\nFormatet e eventDate: {len(t_formats)} formate, {int(is_range.sum())} intervale")
display(t_formats)

u = df.coordinateUncertaintyInMeters
t_unc = u.value_counts().head(10).rename("regjistrime").to_frame()
t_unc["perqindja_e_vlerave_te_dhena"] = (t_unc.regjistrime / u.notna().sum() * 100).round(2)
t_unc["publikuesi_kryesor"] = [publisher[u == v].mode().iat[0] for v in t_unc.index]
t_unc.index.name = "coordinateUncertaintyInMeters"
g7071 = df_raw[u == GRID_10KM]
print(f"\nVlerat më të shpeshta të coordinateUncertaintyInMeters ({int(u.notna().sum()):,} vlera të dhëna)")
display(t_unc)
print(f"7071 m: {len(g7071):,} regjistrime ({len(g7071) / u.notna().sum() * 100:.2f}% e vlerave të dhëna), "
      f"vetëm {g7071[['decimalLatitude', 'decimalLongitude']].drop_duplicates().shape[0]} pika të ndryshme; "
      f"{g7071.issue.fillna('').str.contains('COORDINATE_ROUNDED').mean() * 100:.2f}% kanë COORDINATE_ROUNDED")
display(publisher[g7071.index].value_counts().rename_axis("publikuesi").rename("regjistrime_me_7071").to_frame())

elev_bins = pd.cut(df.elevation.astype("float"), [-np.inf, 0, ELEV[0], 801, 1501, ELEV[1] + 1, np.inf], right=False,
                   labels=["< 0", "1 – 296", "297 – 800", "801 – 1500", "1501 – 2656", "> 2656"])
elev_bins = elev_bins.cat.add_categories("= 0")
elev_bins[df.elevation == 0] = "= 0"
t_elev = elev_bins.value_counts().reindex(["= 0", "< 0", "1 – 296", "297 – 800", "801 – 1500", "1501 – 2656",
                                           "> 2656"]).rename("regjistrime").to_frame()
t_elev["perqindja"] = (t_elev.regjistrime / df.elevation.notna().sum() * 100).round(2)
elev_out = df_raw.loc[elev & ~df.elevation.between(*ELEV),
                      ["gbifID", "elevation", "decimalLatitude", "decimalLongitude", "municipality", "basisOfRecord"]]
elev_out["publikuesi"] = publisher[elev_out.index]
elev_out = elev_out.sort_values("elevation").set_index("gbifID")
print(f"\nelevation: {int(df.elevation.notna().sum()):,} vlera të dhëna, min {df.elevation.min():g}, "
      f"max {df.elevation.max():g}")
display(t_elev)
print("Lartësitë jashtë 297–2656 m (pa 0):")
display(elev_out)


3.9 VLEFSHMËRIA


Tabela: Rregulla | Rreshta që dështojnë | %


Rregulla,Rreshta që dështojnë,%
"VL1 – Koordinatat brenda kutisë së Kosovës (lat 41.85–43.27, lon 20.01–21.80)",0,0.00
VL2 – Pika ka komunë të Kosovës,0,0.00
VL3 – Pa flamuj GBIF për koordinata të gabuara,0,0.00
VL4 – Koordinata me të paktën 3 shifra dhjetore (≤ ~110 m),784,1.67
VL5 – Viti brenda 1758–2026,0,0.00
VL6 – Viti ≥ 1900 (jo regjistrim historik),16,0.04
VL7 – month 1–12,0,0.00
VL8 – day 1–31,0,0.00
VL9 – year-month-day formon datë reale (p.sh. jo 30 shkurt),0,0.00
VL10 – eventDate lexohet si ISO 8601,0,0.00


% = rreshta që dështojnë / rreshta ku rregulla aplikohet (ku vlera ekziston) × 100

Detajet e kontrolleve


,korniza,kontrolli,kolonat,aplikohen,te_pavlefshme,perqindja_e_vlefshme,veprimi,perqindja_qe_deshtojne
id,,,,,,,,
VL1,V1,Koordinatat brenda kutisë së Kosovës (lat 41.8...,"decimalLatitude, decimalLongitude",46831,0,100.00,largo rreshtin,0.00
VL2,V1,Pika ka komunë të Kosovës,"municipality, decimalLatitude, decimalLongitude",46831,0,100.00,largo rreshtin,0.00
VL3,V1,Pa flamuj GBIF për koordinata të gabuara,issue,46831,0,100.00,largo rreshtin,0.00
VL4,–,Koordinata me të paktën 3 shifra dhjetore (≤ ~...,decimalLatitude,46831,784,98.33,shëno; jo për analizë në nivel fshati,1.67
VL5,V5,Viti brenda 1758–2026,year,44563,0,100.00,vendos NaN,0.00
VL6,A2,Viti ≥ 1900 (jo regjistrim historik),year,44563,16,99.96,shëno si historik,0.04
VL7,V6,month 1–12,month,43964,0,100.00,vendos NaN,0.00
VL8,V6,day 1–31,day,43612,0,100.00,vendos NaN,0.00
VL9,V6,year-month-day formon datë reale (p.sh. jo 30 ...,"year, month, day",43612,0,100.00,vendos NaN,0.00


...

,regjistrime_me_7071
publikuesi,
HabiProt,8122
"National Museum of Natural History, Bulgarian Academy of Sciences",102
"Botanical Garden, University of Valencia",1



elevation: 9,820 vlera të dhëna, min 0, max 3050


,regjistrime,perqindja
elevation,,
= 0,3634,37.01
< 0,0,0.00
1 – 296,4,0.04
297 – 800,3713,37.81
801 – 1500,1330,13.54
1501 – 2656,1134,11.55
> 2656,5,0.05


Lartësitë jashtë 297–2656 m (pa 0):


,elevation,decimalLatitude,decimalLongitude,municipality,basisOfRecord,publikuesi
gbifID,,,,,,
1837879632,100.0,42.666570,21.172400,Prishtinë / Priština,PRESERVED_SPECIMEN,"Museum of Comparative Zoology, Harvard University"
41659662,198.0,42.200000,20.716670,Prizren,OCCURRENCE,USDA Agricultural Research Service (ARS)
5284801271,250.0,42.675509,20.328745,Pejë / Peć,HUMAN_OBSERVATION,HabiProt
5284610427,293.0,42.226778,20.697054,Prizren,HUMAN_OBSERVATION,HabiProt
5794847480,2750.0,42.206670,21.119170,Kaçanik / Kačanik,PRESERVED_SPECIMEN,University of Vienna - Herbarium WU
2028433750,2800.0,42.206670,21.119170,Kaçanik / Kačanik,PRESERVED_SPECIMEN,University of Vienna - Herbarium WU
1830967944,2800.0,42.206670,21.119170,Kaçanik / Kačanik,PRESERVED_SPECIMEN,University of Vienna - Herbarium WU
1831004196,2800.0,42.206670,21.119170,Kaçanik / Kačanik,PRESERVED_SPECIMEN,University of Vienna - Herbarium WU
1830967907,3050.0,42.206670,21.119170,Kaçanik / Kačanik,PRESERVED_SPECIMEN,University of Vienna - Herbarium WU


## 3.10 UNIKALITETI

Kontrollohet `gbifID` dhe numërohen duplikatet logjike me kriterin **species + data + koordinata + vëzhgues** (`recordedBy`). Numri nuk supozohet paraprakisht. 18 grupe nga burime të ndryshme shqyrtohen me dorë, dhe prej tyre propozohet definicioni i duplikatit. Kandidatët ruhen në `processed/duplicate_candidates.csv`. Vendimin final dhe largimin i bën Deliza (20–21 Oct). Përshkrimi është te `reports/unikaliteti.md`.

In [14]:
section("3.10 UNIKALITETI")
PROCESSED_DIR = ROOT / "processed"
PROCESSED_DIR.mkdir(exist_ok=True)

# --- 1. gbifID dhe rreshtat identikë ---
print(f"gbifID unik: {df_raw.gbifID.is_unique}  |  gbifID të përsëritura: {int(df_raw.gbifID.duplicated().sum())}  |  "
      f"rreshta identikë: {int(df_raw.duplicated().sum())}")

# --- 2. Duplikatet logjike: species + data + koordinata + vëzhgues ---
dup = df_raw.assign(
    observer=df_raw.recordedBy.str.strip().str.lower(),
    publisher=publisher,
    date_precision=df_raw.eventDate.str.split("/").str[0].str[:10].str.len().map({4: "vit", 7: "muaj", 10: "ditë"}),
)
DUP_KEYS = ["species", "eventDate", "decimalLatitude", "decimalLongitude", "observer"]
has_key = dup[DUP_KEYS].notna().all(axis=1)
dup = dup[has_key]
dup["dup_group"] = dup.groupby(DUP_KEYS).ngroup()
dup["group_size"] = dup.groupby("dup_group").gbifID.transform("size")
cand = dup[dup.group_size > 1].copy()

# Lloji i grupit (rregullat zbatohen me këtë radhë)
SPECIMEN = {"PRESERVED_SPECIMEN", "MATERIAL_SAMPLE", "MATERIAL_CITATION", "LIVING_SPECIMEN", "FOSSIL_SPECIMEN"}
g = cand.groupby("dup_group")
info = pd.DataFrame({
    "n_dataset": g.datasetKey.nunique(),
    "jo_ditore": g.date_precision.agg(lambda s: (s != "ditë").any()),
    "ekzemplare": g.apply(lambda x: x.basisOfRecord.isin(SPECIMEN).all()
                          and x.catalogNumber.notna().all() and x.catalogNumber.nunique() == len(x)),
    "rrjete_10km": g.coordinateUncertaintyInMeters.agg(lambda s: (s == GRID_10KM).any()),
    "numer_i_njejte": g.individualCount.agg(lambda s: s.nunique(dropna=False) == 1),
})
info["lloji"] = np.select(
    [info.n_dataset > 1, info.jo_ditore, info.ekzemplare, info.rrjete_10km, info.numer_i_njejte],
    ["i njëjti rresht në dataset-e të ndryshme", "datë jo ditore (vit/muaj)", "ekzemplarë të ndryshëm (catalogNumber)",
     "qendër e rrjetës 10 km (7071 m)", "vëzhgim i përsëritur me numër të njëjtë"],
    default="numërime të ndryshme të së njëjtës ditë")
DUPLICATE_TYPES = {"i njëjti rresht në dataset-e të ndryshme", "vëzhgim i përsëritur me numër të njëjtë"}
info["duplikat_i_propozuar"] = info.lloji.isin(DUPLICATE_TYPES)
cand = cand.join(info[["lloji", "duplikat_i_propozuar"]], on="dup_group")

by_type = (cand.groupby(["lloji", "duplikat_i_propozuar"])
           .agg(grupe=("dup_group", "nunique"), rreshta=("gbifID", "size"))
           .reset_index())
by_type["rreshta_te_teperta"] = by_type.rreshta - by_type.grupe
by_type = by_type.sort_values(["duplikat_i_propozuar", "rreshta"], ascending=[False, False]).set_index("lloji")


def dup_summary(frame, keys):
    x = frame[frame[keys].notna().all(axis=1)]
    gs = x.groupby(keys).size()
    return int((gs > 1).sum()), int(x.duplicated(keys, keep=False).sum()), int(x.duplicated(keys).sum())


crit = {
    "gbifID": (0, 0, int(df_raw.gbifID.duplicated().sum())),
    "rreshta identikë (të 42 kolonat)": (0, 0, int(df_raw.duplicated().sum())),
    "species + data + koordinata (pa vëzhgues)": dup_summary(df_raw, DUP_KEYS[:4]),
    "species + data + koordinata + vëzhgues (kriteri i planit)": (
        cand.dup_group.nunique(), len(cand), len(cand) - cand.dup_group.nunique()),
    "definicioni i propozuar": (
        int(info.duplikat_i_propozuar.sum()), int(cand.duplikat_i_propozuar.sum()),
        int(cand.duplikat_i_propozuar.sum() - info.duplikat_i_propozuar.sum())),
}
unique_table = pd.DataFrame(crit, index=["grupe", "rreshta_ne_grupe", "rreshta_te_teperta"]).T
unique_table["perqindja_e_teperta"] = (unique_table.rreshta_te_teperta / len(df_raw) * 100).round(2)
unique_table.index.name = "kriteri"
print("\nNumri real i duplikateve sipas kriterit")
display(unique_table)
print("rreshta të tepërta = rreshta në grupe − grupe (një rresht mbetet për çdo grup)")
print(f"Kriteri i planit aplikohet te {int(has_key.sum()):,} rreshta me të 5 fushat të plota "
      f"({int(df_raw.recordedBy.isna().sum()):,} pa recordedBy).")
print("\nGrupet sipas llojit")
display(by_type)

# --- 3. Shqyrtimi manual: 18 raste nga burime dhe lloje të ndryshme ---
REVIEW = {
    5688208496: ("eBird, Garrulus glandarius, Liqeni i Badovcit", "duplikat",
                 "I njëjti vëzhgues, vend dhe ditë, numri 1 në dy lista (OBS të ndryshme): i njëjti shpend në dy lista."),
    5304671044: ("eBird, Picus viridis, Bozevce", "duplikat",
                 "Dy lista të të njëjtit vëzhgues në të njëjtën ditë; numri 1 te të dyja."),
    5505753852: ("eBird, Poecile palustris, Bozevce", "duplikat",
                 "Të njëjtat dy lista si rasti i mësipërm, numri 2 te të dyja: lista e përsëritur, jo shpendë të tjerë."),
    6447736029: ("Observation.org, Phoenicurus ochruros, Mitrovicë", "duplikat",
                 "I njëjti përdorues, pikë (35 m) dhe ditë, numri 1 te të dy: vëzhgimi i ngarkuar dy herë."),
    3071190713: ("Macedonian Ecological Society, Nebria attemsi, Ljuboten 1906", "duplikat",
                 "4 rreshta pa asnjë dallim (pa numër, pa catalogNumber): i njëjti grumbullim i shënuar disa herë."),
    6434425548: ("ENA + BOLD, Euscorpius hadzii, 2021", "duplikat",
                 "S00009 dhe UP SC00009: i njëjti ekzemplar në dy databaza."),
    1584382204: ("Herbarium WU + NHM Wien, Paramoltkia doerfleri, 1918", "duplikat",
                 "Dörfler nr. 848 në dy herbariume: dublikatë herbariumi të të njëjtit grumbullim."),
    6448581086: ("Observation.org, Chlidonias niger, Liqeni 1", "jo duplikat",
                 "I njëjti vend dhe ditë, por 15 dhe 20 individë: dy numërime të ndara."),
    5284987599: ("HabiProt, Caliaeschna microstigma", "jo duplikat",
                 "7 dhe 10 individë, koordinata = qendra e qelizës 10 km: ka gjasa dy vende brenda qelizës."),
    5284716226: ("HabiProt, Carpocoris purpureipennis", "jo duplikat",
                 "4 rreshta me 10, 1, 1, 1 individë në qelizën 10 km: vende ose numërime të ndryshme."),
    5284977542: ("HabiProt, Everes sebrus", "jo duplikat",
                 "Numri 1 te të dy, por pa koordinata të sakta (7071 m) nuk dihet nëse është i njëjti vend; mbahen."),
    3071190766: ("Macedonian Ecological Society, Otiorhynchus relictus, 1906", "jo duplikat",
                 "8 rreshta me 1 ose 2 individë: ekzemplarë ose burime literature të ndryshme."),
    6471379490: ("ENA, Rhyacophila fasciata, 2024", "jo duplikat",
                 "MNCN-ADN 200133 dhe 200132: dy ekzemplarë të ndryshëm nga i njëjti grumbullim."),
    6471376553: ("ENA, Rhyacophila macedonica, 2023", "jo duplikat",
                 "MNCN-ADN 200131 dhe 200130: dy ekzemplarë të ndryshëm."),
    3478785497: ("Adam Mickiewicz Univ., Alinda fallax, 1935-06-01", "jo duplikat",
                 "16 ekzemplarë me catalogNumber të njëpasnjëshëm: individë të ndryshëm të një grumbullimi."),
    3478937662: ("Adam Mickiewicz Univ., Alinda biplicata, 1906", "jo duplikat",
                 "24 ekzemplarë të ndryshëm; data ka vetëm vitin."),
    6191368728: ("ENA, Phlebotomus perfiliewi, 2022", "jo duplikat",
                 "Data vetëm me vit dhe ekzemplarë të ndryshëm (77_63, 07_3)."),
    6191513544: ("ENA, Phlebotomus tobbi, 2022", "jo duplikat",
                 "5 ekzemplarë të ndryshëm; data vetëm me vit."),
}
review = []
for gid_, (rasti, vendimi, arsyeja) in REVIEW.items():
    row = cand[cand.gbifID == gid_]
    assert len(row) == 1, gid_
    r = row.iloc[0]
    review.append({"rasti": rasti, "gbifID": gid_, "rreshta_ne_grup": int(r.group_size),
                   "vendimi_manual": vendimi, "lloji_automatik": r.lloji,
                   "rregulla_perputhet": (vendimi == "duplikat") == bool(r.duplikat_i_propozuar),
                   "arsyeja": arsyeja})
review = pd.DataFrame(review).set_index("rasti")
print(f"\nShqyrtimi manual: {len(review)} raste, {int((review.vendimi_manual == 'duplikat').sum())} duplikatë")
display(review.style.set_properties(subset=["arsyeja"], **{"text-align": "left"}))
print(f"Rregulla automatike përputhet me vendimin manual te {int(review.rregulla_perputhet.sum())} / {len(review)} raste")
assert review.rregulla_perputhet.all()

# --- 4. processed/duplicate_candidates.csv ---
duplicate_candidates = (cand[["dup_group", "group_size", "gbifID", "species", "eventDate", "date_precision",
                              "decimalLatitude", "decimalLongitude", "recordedBy", "individualCount",
                              "catalogNumber", "basisOfRecord", "coordinateUncertaintyInMeters", "datasetKey",
                              "publisher", "lloji", "duplikat_i_propozuar"]]
                        .sort_values(["dup_group", "gbifID"]))
duplicate_candidates.to_csv(PROCESSED_DIR / "duplicate_candidates.csv", index=False)
print(f"\nprocessed/duplicate_candidates.csv: {len(duplicate_candidates):,} rreshta në "
      f"{duplicate_candidates.dup_group.nunique():,} grupe")


3.10 UNIKALITETI
gbifID unik: True  |  gbifID të përsëritura: 0  |  rreshta identikë: 0



Numri real i duplikateve sipas kriterit


,grupe,rreshta_ne_grupe,rreshta_te_teperta,perqindja_e_teperta
kriteri,,,,
gbifID,0,0,0,0.00
rreshta identikë (të 42 kolonat),0,0,0,0.00
species + data + koordinata (pa vëzhgues),1927,4592,2665,5.69
species + data + koordinata + vëzhgues (kriteri i planit),1288,3132,1844,3.94
definicioni i propozuar,238,616,378,0.81


rreshta të tepërta = rreshta në grupe − grupe (një rresht mbetet për çdo grup)
Kriteri i planit aplikohet te 41,395 rreshta me të 5 fushat të plota (3,853 pa recordedBy).

Grupet sipas llojit


,duplikat_i_propozuar,grupe,rreshta,rreshta_te_teperta
lloji,,,,
vëzhgim i përsëritur me numër të njëjtë,True,236,612,376
i njëjti rresht në dataset-e të ndryshme,True,2,4,2
numërime të ndryshme të së njëjtës ditë,False,735,1510,775
qendër e rrjetës 10 km (7071 m),False,239,708,469
ekzemplarë të ndryshëm (catalogNumber),False,51,169,118
datë jo ditore (vit/muaj),False,25,129,104



Shqyrtimi manual: 18 raste, 7 duplikatë


,gbifID,rreshta_ne_grup,vendimi_manual,lloji_automatik,rregulla_perputhet,arsyeja
rasti,,,,,,
"eBird, Garrulus glandarius, Liqeni i Badovcit",5688208496,2,duplikat,vëzhgim i përsëritur me numër të njëjtë,True,"I njëjti vëzhgues, vend dhe ditë, numri 1 në dy lista (OBS të ndryshme): i njëjti shpend në dy lista."
"eBird, Picus viridis, Bozevce",5304671044,2,duplikat,vëzhgim i përsëritur me numër të njëjtë,True,Dy lista të të njëjtit vëzhgues në të njëjtën ditë; numri 1 te të dyja.
"eBird, Poecile palustris, Bozevce",5505753852,2,duplikat,vëzhgim i përsëritur me numër të njëjtë,True,"Të njëjtat dy lista si rasti i mësipërm, numri 2 te të dyja: lista e përsëritur, jo shpendë të tjerë."
"Observation.org, Phoenicurus ochruros, Mitrovicë",6447736029,2,duplikat,vëzhgim i përsëritur me numër të njëjtë,True,"I njëjti përdorues, pikë (35 m) dhe ditë, numri 1 te të dy: vëzhgimi i ngarkuar dy herë."
"Macedonian Ecological Society, Nebria attemsi, Ljuboten 1906",3071190713,4,duplikat,vëzhgim i përsëritur me numër të njëjtë,True,"4 rreshta pa asnjë dallim (pa numër, pa catalogNumber): i njëjti grumbullim i shënuar disa herë."
"ENA + BOLD, Euscorpius hadzii, 2021",6434425548,2,duplikat,i njëjti rresht në dataset-e të ndryshme,True,S00009 dhe UP SC00009: i njëjti ekzemplar në dy databaza.
"Herbarium WU + NHM Wien, Paramoltkia doerfleri, 1918",1584382204,2,duplikat,i njëjti rresht në dataset-e të ndryshme,True,Dörfler nr. 848 në dy herbariume: dublikatë herbariumi të të njëjtit grumbullim.
"Observation.org, Chlidonias niger, Liqeni 1",6448581086,2,jo duplikat,numërime të ndryshme të së njëjtës ditë,True,"I njëjti vend dhe ditë, por 15 dhe 20 individë: dy numërime të ndara."
"HabiProt, Caliaeschna microstigma",5284987599,2,jo duplikat,qendër e rrjetës 10 km (7071 m),True,"7 dhe 10 individë, koordinata = qendra e qelizës 10 km: ka gjasa dy vende brenda qelizës."


Rregulla automatike përputhet me vendimin manual te 18 / 18 raste

processed/duplicate_candidates.csv: 3,132 rreshta në 1,288 grupe


## 3.11 KONSISTENCA DHE ANSHMËRIA

Konsistenca: **species → genus → family** dhe **municipality → district**, edhe kundrejt ndarjes zyrtare të Kosovës (7 rajone, 38 komuna). Anshmëria: % e regjistrimeve dhe e specieve sipas **klasës, rajonit dhe burimit**, plus përqendrimi te vëzhguesi dhe pika. Paragrafi për anshmërinë gjenerohet nga numrat, që të mbetet i saktë kur notebook-u ri-ekzekutohet mbi të dhënat e pastra. Përshkrimi është te `reports/konsistenca_dhe_anshmeria.md`.

In [15]:
section("3.11 KONSISTENCA DHE ANSHMËRIA")
from IPython.display import Markdown

# --- 1. Konsistenca: species → genus → family dhe municipality → district ---
# Ndarja zyrtare e Kosovës: 7 rajone, 38 komuna (emri shqip para " / ")
OFFICIAL_DISTRICTS = {
    "Ferizaj": ["Ferizaj", "Hani i Elezit", "Kaçanik", "Shtime", "Shtërpcë"],
    "Gjakovë": ["Deçan", "Gjakovë", "Junik", "Rahovec"],
    "Gjilan": ["Gjilan", "Kamenicë", "Kllokot", "Partesh", "Ranillug", "Viti"],
    "Mitrovicë": ["Leposaviq", "Mitrovicë e Jugut", "Mitrovicë e Veriut", "Skenderaj", "Vushtrri", "Zubin Potok",
                  "Zveçan"],
    "Pejë": ["Istog", "Klinë", "Pejë"],
    "Prishtinë": ["Fushë Kosovë", "Gllogoc", "Graçanicë", "Lipjan", "Novobërdë", "Obiliq", "Podujevë", "Prishtinë"],
    "Prizren": ["Dragash", "Malishevë", "Mamushë", "Prizren", "Suharekë"],
}
official = {m: d for d, ms in OFFICIAL_DISTRICTS.items() for m in ms}
alb = lambda s: s.str.split(" / ").str[0]


def hierarchy_check(child, parent, within=()):
    keys = [*within, child]
    x = df_raw.dropna(subset=keys + [parent])
    n_parents = x.groupby(keys)[parent].transform("nunique")
    bad = x[n_parents > 1]
    examples = (bad.groupby(keys)[parent].unique().head(3)
                .apply(lambda v: " / ".join(sorted(v))))
    ex = "; ".join(f"{k if isinstance(k, str) else k[-1]} → {v}" for k, v in examples.items())
    return {"vlera": x.groupby(keys).ngroups, "vlera_jokonsistente": bad.groupby(keys).ngroups,
            "aplikohen": len(x), "rreshta_jokonsistente": len(bad), "shembuj": ex}


sp = df_raw.dropna(subset=["species", "genus"])
name_bad = sp[sp.species.str.split().str[0] != sp.genus]
mun_district = df_raw[["municipality", "district"]].drop_duplicates()
off_bad = mun_district[alb(mun_district.municipality).map(official) != alb(mun_district.district)]
consistency = pd.DataFrame({
    "species → genus (çdo specie ka një gjini)": hierarchy_check("species", "genus"),
    "species → genus (fjala e parë e species = genus)": {
        "vlera": sp.species.nunique(), "vlera_jokonsistente": name_bad.species.nunique(), "aplikohen": len(sp),
        "rreshta_jokonsistente": len(name_bad),
        "shembuj": "; ".join(f"{s} → {g}" for s, g in name_bad.groupby("species").genus.first().head(3).items())},
    "genus → family (pa kusht)": hierarchy_check("genus", "family"),
    "genus → family (brenda kingdom)": hierarchy_check("genus", "family", within=["kingdom"]),
    "family → order → class → phylum → kingdom": {
        k: sum(hierarchy_check(c, p, ["kingdom"] if p != "kingdom" else [])[k]
               for c, p in [("family", "order"), ("order", "class"), ("class", "phylum"), ("phylum", "kingdom")])
        if k != "shembuj" else "" for k in ["vlera", "vlera_jokonsistente", "aplikohen", "rreshta_jokonsistente", "shembuj"]},
    "municipality → district (çdo komunë një rajon)": hierarchy_check("municipality", "district"),
    "municipality → district (sipas ndarjes zyrtare, 7 rajone / 38 komuna)": {
        "vlera": len(mun_district), "vlera_jokonsistente": len(off_bad), "aplikohen": len(df_raw),
        "rreshta_jokonsistente": int(df_raw.municipality.isin(off_bad.municipality).sum()), "shembuj": ""},
}).T
int_cols = ["vlera", "vlera_jokonsistente", "aplikohen", "rreshta_jokonsistente"]
consistency[int_cols] = consistency[int_cols].astype(int)
consistency["perqindja_konsistente"] = (
    (1 - consistency.rreshta_jokonsistente.astype(int) / consistency.aplikohen.astype(int)) * 100).round(2)
consistency.index.name = "rregulla"
consistency = consistency[["vlera", "vlera_jokonsistente", "aplikohen", "rreshta_jokonsistente",
                           "perqindja_konsistente", "shembuj"]]
print("Tabela e konsistencës")
display(consistency.style.set_properties(subset=["shembuj"], **{"text-align": "left"}))
print(f"Komuna në dataset: {df_raw.municipality.nunique()} / 38  |  rajone: {df_raw.district.nunique()} / 7")

# --- 2. Anshmëria: % sipas klasës, rajonit dhe burimit ---
N = len(df_raw)
n_species = df_raw.species.nunique()
henc = (df_raw.decimalLatitude == 42.579243) & (df_raw.decimalLongitude == 21.048304)  # Ligatinat e Henc (eBird)


def share_table(col, top=None):
    g = df_raw.groupby(df_raw[col].fillna("(mungon)"))
    t = pd.DataFrame({"regjistrime": g.size(), "specie": g.species.nunique()}).sort_values("regjistrime",
                                                                                           ascending=False)
    if top and len(t) > top:
        rest = df_raw[~df_raw[col].fillna("(mungon)").isin(t.index[:top])]
        t = pd.concat([t.iloc[:top], pd.DataFrame({"regjistrime": [len(rest)], "specie": [rest.species.nunique()]},
                                                  index=[f"të tjera ({len(t) - top})"])])
    t["perqindja_regjistrimeve"] = (t.regjistrime / N * 100).round(2)
    t["perqindja_specieve"] = (t.specie / n_species * 100).round(2)
    t.index.name = col
    return t


bias_class = share_table("class", top=10)
bias_class["regjistrime_per_specie"] = (bias_class.regjistrime / bias_class.specie).round(1)

bias_district = share_table("district")
bias_district["perqindja_pa_henc"] = (df_raw[~henc].district.value_counts() / (~henc).sum() * 100).round(2)
bias_district["regjistrime_per_specie"] = (bias_district.regjistrime / bias_district.specie).round(1)

def main_class(s):
    vc = s.value_counts()
    return f"{vc.index[0]} ({vc.iloc[0] / len(s) * 100:.0f}%)" if len(vc) else "(pa klasë)"


df_pub = df_raw.assign(publisher=publisher)
g = df_pub.groupby("publisher")
bias_source = pd.DataFrame({"regjistrime": g.size(), "specie": g.species.nunique(),
                            "klasa_kryesore": g["class"].agg(main_class)}).sort_values("regjistrime", ascending=False)
rest = bias_source.iloc[10:]
bias_source = pd.concat([bias_source.iloc[:10], pd.DataFrame(
    {"regjistrime": [rest.regjistrime.sum()],
     "specie": [df_pub[df_pub.publisher.isin(rest.index)].species.nunique()], "klasa_kryesore": [""]},
    index=[f"të tjerë ({len(rest)})"])])
bias_source["perqindja_regjistrimeve"] = (bias_source.regjistrime / N * 100).round(2)
bias_source["perqindja_specieve"] = (bias_source.specie / n_species * 100).round(2)
bias_source.index.name = "publikuesi"

print("\nAnshmëria sipas klasës (10 klasat kryesore)")
display(bias_class)
print("Anshmëria sipas rajonit")
display(bias_district)
print("Anshmëria sipas burimit (10 publikuesit kryesorë)")
display(bias_source)


def top_share(s):
    vc = s.value_counts()
    return pd.Series({"vlera_me_e_shpeshte": vc.index[0], "perqindja": round(vc.iloc[0] / N * 100, 2),
                      "top_3_perqindja": round(vc.iloc[:3].sum() / N * 100, 2), "vlera_te_ndryshme": len(vc)})


concentration = pd.DataFrame({
    "klasa": top_share(df_raw["class"]),
    "rajoni": top_share(df_raw.district),
    "komuna": top_share(df_raw.municipality),
    "publikuesi": top_share(publisher),
    "vëzhguesi (recordedBy)": top_share(df_raw.recordedBy),
    "pika (koordinatat)": top_share(df_raw.decimalLatitude.astype(str) + ", " + df_raw.decimalLongitude.astype(str)),
}).T
concentration.index.name = "dimensioni"
print("Përqendrimi: sa regjistrime jep vlera më e shpeshtë")
display(concentration)

# --- 3. Paragrafi për anshmërinë (numrat llogariten nga tabelat më lart) ---
c = bias_class.loc
d = bias_district.loc
top_obs = df_raw.recordedBy.value_counts()
ebird = "Cornell Lab of Ornithology"
aves_from_ebird = (publisher[df_raw["class"] == "Aves"] == ebird).mean() * 100
richest = bias_district.specie.idxmax()
bias_paragraph = (
    f"**Anshmëria.** Dataseti nuk është mostër e rastësishme e biodiversitetit të Kosovës. Ai pasqyron se ku "
    f"dhe çka vëzhgojnë disa vëzhgues dhe platforma. **Sipas klasës**, zogjtë (Aves) japin "
    f"{c['Aves', 'perqindja_regjistrimeve']:.2f}% të regjistrimeve, por vetëm {c['Aves', 'perqindja_specieve']:.2f}% "
    f"të specieve. Insektet japin {c['Insecta', 'perqindja_regjistrimeve']:.2f}% të regjistrimeve dhe "
    f"{c['Insecta', 'perqindja_specieve']:.2f}% të specieve, kurse bimët me lule (Magnoliopsida) "
    f"{c['Magnoliopsida', 'perqindja_regjistrimeve']:.2f}% dhe {c['Magnoliopsida', 'perqindja_specieve']:.2f}%. "
    f"**Sipas rajonit**, rajoni i Prishtinës ka {d['Prishtinë / Priština', 'perqindja_regjistrimeve']:.2f}% të "
    f"regjistrimeve, dhe vetëm komuna e Graçanicës "
    f"{concentration.loc['komuna', 'perqindja']:.2f}%. Arsyeja është që {henc.mean() * 100:.2f}% e të gjithë "
    f"datasetit vjen nga një pikë e vetme, hotspot-i i eBird te Ligatinat e Henc. Pa këtë pikë, pjesa e rajonit "
    f"bie në {d['Prishtinë / Priština', 'perqindja_pa_henc']:.2f}%. Megjithatë, më shumë specie ka rajoni i "
    f"{richest.split(' / ')[0]}it ({int(d[richest, 'specie']):,} specie), me vetëm "
    f"{d[richest, 'perqindja_regjistrimeve']:.2f}% të regjistrimeve. **Sipas burimit**, eBird jep "
    f"{bias_source.loc[ebird, 'perqindja_regjistrimeve']:.2f}% të regjistrimeve dhe {aves_from_ebird:.1f}% të "
    f"regjistrimeve të zogjve. Një vëzhgues i vetëm ({top_obs.index[0]}) ka {top_obs.iloc[0] / N * 100:.2f}% "
    f"të të gjithë regjistrimeve. Prandaj numri i regjistrimeve për komunë, rajon ose klasë mat kryesisht "
    f"përpjekjen e vëzhgimit, jo bollëkun ose pasurinë reale. Krahasimet duhet të mbështeten te numri i specieve "
    f"unike ose te vlera të normalizuara (p.sh. për km², pas integrimit më 19–20 tetor), dhe rezultatet kryesore "
    f"duhet të kontrollohen edhe pa eBird dhe pa pikën e Henc-it."
)
display(Markdown(bias_paragraph))


3.11 KONSISTENCA DHE ANSHMËRIA


Tabela e konsistencës


,vlera,vlera_jokonsistente,aplikohen,rreshta_jokonsistente,perqindja_konsistente,shembuj
rregulla,,,,,,
species → genus (çdo specie ka një gjini),4095,0,45618,0,100.000000,
species → genus (fjala e parë e species = genus),4095,9,45618,78,99.830000,Dicallomera fascelina → Calliteara; Hippurites archiaci → Vaccinites; Hippurites vesiculosus → Vaccinites
genus → family (pa kusht),2408,3,46615,119,99.740000,Aspicilia → Hymeneliaceae / Megasporaceae; Linaria → Fringillidae / Plantaginaceae; Prunella → Lamiaceae / Prunellidae
genus → family (brenda kingdom),2410,1,46615,4,99.990000,Aspicilia → Hymeneliaceae / Megasporaceae
family → order → class → phylum → kingdom,980,0,185868,0,100.000000,
municipality → district (çdo komunë një rajon),38,0,46831,0,100.000000,
"municipality → district (sipas ndarjes zyrtare, 7 rajone / 38 komuna)",38,0,46831,0,100.000000,


Komuna në dataset: 38 / 38  |  rajone: 7 / 7

Anshmëria sipas klasës (10 klasat kryesore)


,regjistrime,specie,perqindja_regjistrimeve,perqindja_specieve,regjistrime_per_specie
class,,,,,
Aves,28785,255,61.47,6.23,112.9
Insecta,11543,2226,24.65,54.36,5.2
Magnoliopsida,2701,760,5.77,18.56,3.6
Liliopsida,940,143,2.01,3.49,6.6
Lecanoromycetes,612,238,1.31,5.81,2.6
(mungon),388,63,0.83,1.54,6.2
Arachnida,358,63,0.76,1.54,5.7
Gastropoda,279,48,0.60,1.17,5.8
Mammalia,273,51,0.58,1.25,5.4


...

Anshmëria sipas burimit (10 publikuesit kryesorë)


,regjistrime,specie,klasa_kryesore,perqindja_regjistrimeve,perqindja_specieve
publikuesi,,,,,
Cornell Lab of Ornithology,26009,219,Aves (100%),55.54,5.35
HabiProt,8122,1865,Insecta (100%),17.34,45.54
iNaturalist.org,4286,1481,Insecta (29%),9.15,36.17
Observation.org,1436,395,Aves (73%),3.07,9.65
naturgucker.de,721,398,Magnoliopsida (45%),1.54,9.72
Pl@ntNet,660,279,Magnoliopsida (94%),1.41,6.81
University of Graz,659,258,Lecanoromycetes (89%),1.41,6.30
VectorNet,587,36,Insecta (95%),1.25,0.88
Macedonian Ecological Society,483,137,Insecta (74%),1.03,3.35


Përqendrimi: sa regjistrime jep vlera më e shpeshtë


,vlera_me_e_shpeshte,perqindja,top_3_perqindja,vlera_te_ndryshme
dimensioni,,,,
klasa,Aves,61.47,91.88,68
rajoni,Prishtinë / Priština,54.95,75.07,7
komuna,Graçanicë / Gračanica,31.33,52.51,38
publikuesi,Cornell Lab of Ornithology,55.54,82.03,80
vëzhguesi (recordedBy),obsr1505715,35.18,44.21,1519
pika (koordinatat),"42.579243, 21.048304",27.86,36.77,7567


**Anshmëria.** Dataseti nuk është mostër e rastësishme e biodiversitetit të Kosovës. Ai pasqyron se ku dhe çka vëzhgojnë disa vëzhgues dhe platforma. **Sipas klasës**, zogjtë (Aves) japin 61.47% të regjistrimeve, por vetëm 6.23% të specieve. Insektet japin 24.65% të regjistrimeve dhe 54.36% të specieve, kurse bimët me lule (Magnoliopsida) 5.77% dhe 18.56%. **Sipas rajonit**, rajoni i Prishtinës ka 54.95% të regjistrimeve, dhe vetëm komuna e Graçanicës 31.33%. Arsyeja është që 27.86% e të gjithë datasetit vjen nga një pikë e vetme, hotspot-i i eBird te Ligatinat e Henc. Pa këtë pikë, pjesa e rajonit bie në 37.55%. Megjithatë, më shumë specie ka rajoni i Prizrenit (1,582 specie), me vetëm 11.25% të regjistrimeve. **Sipas burimit**, eBird jep 55.54% të regjistrimeve dhe 90.4% të regjistrimeve të zogjve. Një vëzhgues i vetëm (obsr1505715) ka 35.18% të të gjithë regjistrimeve. Prandaj numri i regjistrimeve për komunë, rajon ose klasë mat kryesisht përpjekjen e vëzhgimit, jo bollëkun ose pasurinë reale. Krahasimet duhet të mbështeten te numri i specieve unike ose te vlera të normalizuara (p.sh. për km², pas integrimit më 19–20 tetor), dhe rezultatet kryesore duhet të kontrollohen edhe pa eBird dhe pa pikën e Henc-it.

## 4. RUAJTJA E RAPORTEVE

In [16]:
section("4. RUAJTJA E RAPORTEVE")
REPORTS_DIR.mkdir(exist_ok=True)
missing.to_csv(REPORTS_DIR / "01_vlerat_e_zbrazeta.csv")
quality.to_csv(REPORTS_DIR / "01_kontrollet_e_kualitetit.csv")
issues.to_csv(REPORTS_DIR / "01_flamujt_gbif.csv")
type_report.to_csv(REPORTS_DIR / "01_tipet_e_kolonave.csv")
framework.to_csv(REPORTS_DIR / "01_korniza_e_kualitetit.csv")
col_completeness.to_csv(REPORTS_DIR / "01_plotesia_kolona.csv")
row_completeness.to_csv(REPORTS_DIR / "01_plotesia_rreshta.csv")
validity.to_csv(REPORTS_DIR / "01_vlefshmeria.csv")
rule_table.to_csv(REPORTS_DIR / "01_vlefshmeria_rregullat.csv", index=False)
t_formats.to_csv(REPORTS_DIR / "01_formatet_e_datave.csv")
t_unc.to_csv(REPORTS_DIR / "01_pasiguria_vlerat.csv")
elev_out.to_csv(REPORTS_DIR / "01_lartesia_jashte_diapazonit.csv")
unique_table.to_csv(REPORTS_DIR / "01_unikaliteti.csv")
by_type.to_csv(REPORTS_DIR / "01_duplikatet_sipas_llojit.csv")
review.to_csv(REPORTS_DIR / "01_duplikatet_shqyrtimi.csv")
consistency.to_csv(REPORTS_DIR / "01_konsistenca.csv")
bias_class.to_csv(REPORTS_DIR / "01_anshmeria_klasa.csv")
bias_district.to_csv(REPORTS_DIR / "01_anshmeria_rajoni.csv")
bias_source.to_csv(REPORTS_DIR / "01_anshmeria_burimi.csv")
concentration.to_csv(REPORTS_DIR / "01_anshmeria_perqendrimi.csv")
(REPORTS_DIR / "01_anshmeria_paragrafi.md").write_text(bias_paragraph + "\n", encoding="utf-8")
print("Raportet u ruajtën në", REPORTS_DIR.resolve())


4. RUAJTJA E RAPORTEVE
Raportet u ruajtën në C:\Master\PVDH\reports


## 5. INTEGRIMI: TË DHËNAT E JASHTME (38 KOMUNAT)

Sipërfaqja (km²) dhe popullsia e 38 komunave, për normalizimin e agregimeve (specie/km², regjistrime për 1,000 banorë).

- **Popullsia 2024:** ASK, *Të dhënat demografike sipas komunave – Regjistrimi i Popullsisë 2024* (korrik 2025), Tab. 1.2. Për 4 komunat veriore (Leposaviq, Mitrovicë e Veriut, Zubin Potok, Zveçan) është vlerësim i ASK, sepse regjistrimi u bojkotua. Numri i regjistruar (Tab. 4.1) ruhet veçmas.
- **Sipërfaqja:** matja zyrtare e territorit, 20 shkurt 2017 (Komisioni shtetëror, Agjencia Kadastrale), e njëjta matje që ASK citon për 10,905.25 km². ASK nuk publikon sipërfaqen sipas komunave.

Bashkimi me dataset-in bëhet më 20 tetor. Burimet janë te `reports/burimet_e_te_dhenave_te_komunave.md`.

In [17]:
section("5. INTEGRIMI: TË DHËNAT E JASHTME (38 KOMUNAT)")
# Burimet (shih reports/burimet_e_te_dhenave_te_komunave.md):
SRC_POP = ("ASK – Të dhënat demografike sipas komunave, Regjistrimi i Popullsisë, Ekonomive Familjare dhe "
           "Banesave 2024 (korrik 2025), Tab. 1.2",
           "https://askapi.rks-gov.net/Custom/df9470d5-2575-458e-8521-9023c9b561b4.pdf")
SRC_POP_REG = ("ASK – Rezultatet përfundimtare të Regjistrimit të Popullsisë 2024 (dhjetor 2024), Tab. 4.1",
               "https://askapi.rks-gov.net/Custom/31bc24d2-45e4-4eb5-a567-405f4bdd197f.pdf")
SRC_AREA = ("Komisioni shtetëror për matjen e territorit (KSHMTRK) – Raport faktografik: Matja e territorit të "
            "Republikës së Kosovës (20 shkurt 2017), tabela në f. 52; e njëjta matje që ASK citon për 10.905,25 km²",
            "https://web.archive.org/web/20181209124523/http://kryeministri-ks.net/wp-content/uploads/2018/03/"
            "Raport_Faktografik_Matja_e_territorit_te_Republikes_se_Kosoves_032017.pdf")

# komuna (emri shqip si në dataset) | emri te ASK | popullsia 2024 (Tab. 1.2, me vlerësimin e veriut) |
# popullsia e regjistruar 2024 (Tab. 4.1) | sipërfaqja km² (2017)
MUNICIPALITIES = [
    ("Deçan", "Deçan", 27_775, 27_775, 293.97),
    ("Dragash", "Dragash", 28_896, 28_896, 433.85),
    ("Ferizaj", "Ferizaj", 109_255, 109_255, 344.61),
    ("Fushë Kosovë", "Fushë Kosovë", 63_949, 63_949, 84.09),
    ("Gjakovë", "Gjakovë", 78_699, 78_699, 586.62),
    ("Gjilan", "Gjilan", 82_980, 82_980, 391.84),
    ("Gllogoc", "Gllogoc", 48_079, 48_079, 275.63),
    ("Graçanicë", "Graçanicë", 18_486, 18_486, 122.41),
    ("Hani i Elezit", "Hani i Elezit", 8_533, 8_533, 83.11),
    ("Istog", "Istog", 33_008, 33_008, 454.36),
    ("Junik", "Junik", 3_943, 3_943, 77.78),
    ("Kaçanik", "Kaçanik", 27_716, 27_716, 211.28),
    ("Kamenicë", "Kamenicë", 22_868, 22_868, 416.61),
    ("Klinë", "Klinë", 30_503, 30_503, 309.02),
    ("Kllokot", "Kllokot", 3_041, 3_041, 23.39),
    ("Leposaviq", "Leposaviq", 9_485, 3_185, 539.05),
    ("Lipjan", "Lipjan", 55_044, 55_044, 338.41),
    ("Malishevë", "Malishevë", 43_888, 43_888, 306.42),
    ("Mamushë", "Mamushë", 5_607, 5_607, 10.94),
    ("Mitrovicë e Jugut", "Mitrovicë", 64_742, 64_742, np.nan),
    ("Mitrovicë e Veriut", "Mitrovicë e Veriut", 7_920, 2_326, np.nan),
    ("Novobërdë", "Novobërdë", 4_493, 4_493, 203.98),
    ("Obiliq", "Obiliq", 22_815, 22_815, 104.84),
    ("Partesh", "Partesh", 3_240, 3_240, 28.67),
    ("Pejë", "Pejë", 82_745, 82_745, 602.63),
    ("Podujevë", "Podujevë", 70_975, 70_975, 632.59),
    ("Prishtinë", "Prishtinë", 227_466, 227_466, 523.13),
    ("Prizren", "Prizren", 147_246, 147_246, 626.86),
    ("Rahovec", "Rahovec", 41_799, 41_799, 275.90),
    ("Ranillug", "Ranillug", 2_481, 2_481, 77.62),
    ("Shtërpcë", "Shtërpcë", 10_771, 10_771, 247.70),
    ("Shtime", "Shtime", 24_308, 24_308, 134.42),
    ("Skenderaj", "Skënderaj", 40_664, 40_664, 374.37),
    ("Suharekë", "Suharekë", 45_749, 45_749, 361.04),
    ("Viti", "Viti", 35_566, 35_566, 269.69),
    ("Vushtrri", "Vushtrri", 61_528, 61_528, 344.85),
    ("Zubin Potok", "Zubin Potok", 3_385, 763, 334.38),
    ("Zveçan", "Zveqan", 2_867, 434, 123.01),
]
NORTH_ESTIMATED = {"Leposaviq", "Mitrovicë e Veriut", "Zubin Potok", "Zveçan"}
MITROVICA = ["Mitrovicë e Jugut", "Mitrovicë e Veriut"]
MITROVICA_AREA = 336.18  # matja e vitit 2017 i jep të dyja komunat bashkë

mun = pd.DataFrame(MUNICIPALITIES, columns=["komuna", "emri_ask", "popullsia_2024", "popullsia_e_regjistruar_2024",
                                            "siperfaqja_km2"])
mun.insert(1, "rajoni", mun.komuna.map(official))
mun["popullsia_vleresim_ask"] = mun.komuna.isin(NORTH_ESTIMATED)
mun["njesia_e_siperfaqes"] = np.where(mun.komuna.isin(MITROVICA), " + ".join(MITROVICA), mun.komuna)
mun["siperfaqja_njesise_km2"] = np.where(mun.komuna.isin(MITROVICA), MITROVICA_AREA, mun.siperfaqja_km2)
unit = mun.groupby("njesia_e_siperfaqes").agg(pop=("popullsia_2024", "sum"), area=("siperfaqja_njesise_km2", "first"))
mun["dendesia_banore_km2"] = mun.njesia_e_siperfaqes.map((unit["pop"] / unit["area"]).round(1))
mun["burimi_popullsise"], mun["linku_popullsise"], mun["viti_popullsise"] = SRC_POP[0], SRC_POP[1], 2024
mun["burimi_siperfaqes"], mun["linku_siperfaqes"], mun["viti_siperfaqes"] = SRC_AREA[0], SRC_AREA[1], 2017

# --- Kontrollet ---
dataset_mun = set(alb(df_raw.municipality.drop_duplicates()))
area_total = round(mun.drop_duplicates("njesia_e_siperfaqes").siperfaqja_njesise_km2.sum(), 2)
checks_mun = pd.Series({
    "38 komuna, pa përsëritje": len(mun) == 38 and mun.komuna.is_unique,
    "7 rajone sipas ndarjes zyrtare": mun.rajoni.notna().all() and mun.rajoni.nunique() == 7,
    "të njëjtat 38 komuna si në dataset": set(mun.komuna) == dataset_mun,
    "popullsia gjithsej = 1,602,515 (ASK, me vlerësimin e veriut)": mun.popullsia_2024.sum() == 1_602_515,
    "popullsia e regjistruar = 1,585,566 (ASK, regjistrimi)": mun.popullsia_e_regjistruar_2024.sum() == 1_585_566,
    "sipërfaqja gjithsej = 10,905.25 km² (ASK / Agjencia Kadastrale)": area_total == 10_905.25,
    "regjistruar = gjithsej jashtë 4 komunave veriore": (
        mun.loc[~mun.popullsia_vleresim_ask, "popullsia_2024"]
        == mun.loc[~mun.popullsia_vleresim_ask, "popullsia_e_regjistruar_2024"]).all(),
}, name="kalon")
display(checks_mun.to_frame())
assert checks_mun.all()

show_cols = ["komuna", "rajoni", "popullsia_2024", "popullsia_e_regjistruar_2024", "popullsia_vleresim_ask",
             "siperfaqja_km2", "siperfaqja_njesise_km2", "dendesia_banore_km2"]
print(f"\n38 komuna | popullsia {mun.popullsia_2024.sum():,} | sipërfaqja {area_total:,.2f} km² | "
      f"dendësia mesatare {mun.popullsia_2024.sum() / area_total:.1f} banorë/km²")
display(mun[show_cols].sort_values("komuna").reset_index(drop=True))
print("Mitrovicë e Jugut dhe e Veriut: matja e vitit 2017 i jep bashkë (336.18 km²), prandaj sipërfaqja e "
      "secilës mbetet bosh dhe dendësia llogaritet për të dyja bashkë.")
print("Leposaviq, Mitrovicë e Veriut, Zubin Potok, Zveçan: popullsia_2024 është vlerësim i ASK (bojkot i "
      "regjistrimit); popullsia_e_regjistruar_2024 është numri i regjistruar.")

mun.to_csv(PROCESSED_DIR / "municipalities.csv", index=False)
print(f"\nprocessed/municipalities.csv: {len(mun)} rreshta, {mun.shape[1]} kolona")


5. INTEGRIMI: TË DHËNAT E JASHTME (38 KOMUNAT)


,kalon
"38 komuna, pa përsëritje",True
7 rajone sipas ndarjes zyrtare,True
të njëjtat 38 komuna si në dataset,True
"popullsia gjithsej = 1,602,515 (ASK, me vlerësimin e veriut)",True
"popullsia e regjistruar = 1,585,566 (ASK, regjistrimi)",True
"sipërfaqja gjithsej = 10,905.25 km² (ASK / Agjencia Kadastrale)",True
regjistruar = gjithsej jashtë 4 komunave veriore,True



38 komuna | popullsia 1,602,515 | sipërfaqja 10,905.25 km² | dendësia mesatare 146.9 banorë/km²


,komuna,rajoni,popullsia_2024,popullsia_e_regjistruar_2024,popullsia_vleresim_ask,siperfaqja_km2,siperfaqja_njesise_km2,dendesia_banore_km2
0,Deçan,Gjakovë,27775,27775,False,293.97,293.97,94.5
1,Dragash,Prizren,28896,28896,False,433.85,433.85,66.6
2,Ferizaj,Ferizaj,109255,109255,False,344.61,344.61,317.0
3,Fushë Kosovë,Prishtinë,63949,63949,False,84.09,84.09,760.5
4,Gjakovë,Gjakovë,78699,78699,False,586.62,586.62,134.2
5,Gjilan,Gjilan,82980,82980,False,391.84,391.84,211.8
6,Gllogoc,Prishtinë,48079,48079,False,275.63,275.63,174.4
7,Graçanicë,Prishtinë,18486,18486,False,122.41,122.41,151.0
8,Hani i Elezit,Ferizaj,8533,8533,False,83.11,83.11,102.7
9,Istog,Pejë,33008,33008,False,454.36,454.36,72.6


Mitrovicë e Jugut dhe e Veriut: matja e vitit 2017 i jep bashkë (336.18 km²), prandaj sipërfaqja e secilës mbetet bosh dhe dendësia llogaritet për të dyja bashkë.
Leposaviq, Mitrovicë e Veriut, Zubin Potok, Zveçan: popullsia_2024 është vlerësim i ASK (bojkot i regjistrimit); popullsia_e_regjistruar_2024 është numri i regjistruar.

processed/municipalities.csv: 38 rreshta, 16 kolona


## 5.2 INTEGRIMI: BASHKIMI ME DATASET-IN

**Çelësi i komunës** krijohet nga emri dygjuhësh: pjesa shqip para ` / `, me shkronja të vogla dhe pa diakritikë (`Graçanicë / Gračanica` → `gracanice`). Dataset-i bashkohet me `processed/municipalities.csv` me **left join** (`validate="many_to_one"`). Kontrollohet: 38/38 komuna përputhen dhe numri i rreshtave është i njëjtë para dhe pas. Rezultati ruhet në `processed/biodiversity_municipalities.csv`. Përshkrimi është te `reports/bashkimi_i_komunave.md`.

In [18]:
section("5.2 INTEGRIMI: BASHKIMI ME DATASET-IN")
import unicodedata


def municipality_key(name):
    """Çelësi i komunës: pjesa shqip para ' / ', me shkronja të vogla, pa diakritikë dhe pa hapësira të tepërta.
    'Ferizaj / Uroševac' -> 'ferizaj', 'Graçanicë / Gračanica' -> 'gracanice'."""
    s = str(name).split(" / ")[0].strip().lower()
    s = "".join(ch for ch in unicodedata.normalize("NFKD", s) if not unicodedata.combining(ch))
    return " ".join(s.split())


municipalities = pd.read_csv(PROCESSED_DIR / "municipalities.csv")
municipalities["komuna_key"] = municipalities.komuna.map(municipality_key)
assert municipalities.komuna_key.is_unique

JOIN_COLS = ["komuna_key", "popullsia_2024", "popullsia_e_regjistruar_2024", "popullsia_vleresim_ask",
             "siperfaqja_km2", "njesia_e_siperfaqes", "siperfaqja_njesise_km2", "dendesia_banore_km2"]
left = df_raw.assign(komuna_key=df_raw.municipality.map(municipality_key))
joined = left.merge(municipalities[JOIN_COLS], on="komuna_key", how="left", validate="many_to_one",
                    indicator=True)

# --- Tabela e çelësave: emri dygjuhësh -> çelësi -> komuna te municipalities.csv ---
key_table = (left.groupby(["municipality", "komuna_key"]).size().rename("rreshta").reset_index()
             .merge(municipalities[["komuna_key", "komuna", "emri_ask"]], on="komuna_key", how="outer",
                    indicator="perputhja")
             .sort_values("komuna_key").set_index("komuna_key"))
key_table["perputhja"] = key_table.perputhja.map({"both": "po", "left_only": "vetëm në dataset",
                                                  "right_only": "vetëm te municipalities.csv"})
key_table["emri_ask_perputhet"] = key_table.emri_ask.map(municipality_key) == key_table.index
print("Çelësi i komunës për 38 emrat")
display(key_table)

# --- Kontrollet e bashkimit ---
mun_rows = joined.drop_duplicates("komuna_key")
join_check = pd.Series({
    "rreshta para": len(df_raw),
    "rreshta pas": len(joined),
    "kolona para": df_raw.shape[1],
    "kolona pas": joined.shape[1] - 1,  # pa kolonën ndihmëse _merge
    "komuna në dataset": left.komuna_key.nunique(),
    "komuna te municipalities.csv": len(municipalities),
    "komuna që përputhen": int((key_table.perputhja == "po").sum()),
    "rreshta pa përputhje": int((joined._merge != "both").sum()),
    "rreshta me popullsi bosh pas bashkimit": int(joined.popullsia_2024.isna().sum()),
    "rreshta me sipërfaqe njësie bosh pas bashkimit": int(joined.siperfaqja_njesise_km2.isna().sum()),
    "popullsia e 38 komunave pas bashkimit": int(mun_rows.popullsia_2024.sum()),
    "emra të ASK që përputhen pa korrigjim": int(key_table.emri_ask_perputhet.sum()),
}, name="vlera").to_frame()
join_check["pritet"] = [len(df_raw), len(df_raw), df_raw.shape[1], df_raw.shape[1] + len(JOIN_COLS), 38, 38, 38,
                        0, 0, 0, 1_602_515, "–"]
join_check["kalon"] = [v == p if p != "–" else "–" for v, p in zip(join_check.vlera, join_check.pritet)]
print("\nRezultati i kontrollit të bashkimit")
display(join_check)
assert all(k for k in join_check.kalon if k != "–"), "Bashkimi nuk i kaloi kontrollet"
assert joined.gbifID.equals(df_raw.gbifID), "Rendi ose numri i rreshtave ndryshoi"

key_table.to_csv(REPORTS_DIR / "01_bashkimi_celesat.csv")
join_check.to_csv(REPORTS_DIR / "01_bashkimi_kontrolli.csv")

# Kolonat origjinale ruhen tekstualisht si në CSV-në burimore (p.sh. 2026, TRUE), pa konvertimet e pandas
raw_text = pd.read_csv(DATA_PATH, dtype=str, keep_default_na=False)
biodiversity_municipalities = pd.concat([raw_text, joined[JOIN_COLS].reset_index(drop=True)], axis=1)
assert biodiversity_municipalities.gbifID.astype("int64").equals(df_raw.gbifID)
OUT_JOINED = PROCESSED_DIR / "biodiversity_municipalities.csv"
biodiversity_municipalities.to_csv(OUT_JOINED, index=False)
print(f"\n{OUT_JOINED.relative_to(ROOT)}: {len(biodiversity_municipalities):,} rreshta, "
      f"{biodiversity_municipalities.shape[1]} kolona ({df_raw.shape[1]} origjinale + {len(JOIN_COLS)} të reja), "
      f"{OUT_JOINED.stat().st_size / 1024**2:.1f} MB")
print("Kolonat e reja:", ", ".join(JOIN_COLS))


5.2 INTEGRIMI: BASHKIMI ME DATASET-IN


Çelësi i komunës për 38 emrat


,municipality,rreshta,komuna,emri_ask,perputhja,emri_ask_perputhet
komuna_key,,,,,,
decan,Deçan / Dečan,1403,Deçan,Deçan,po,True
dragash,Dragash / Dragaš,939,Dragash,Dragash,po,True
ferizaj,Ferizaj / Uroševac,337,Ferizaj,Ferizaj,po,True
fushe kosove,Fushë Kosovë / Kosovo Polje,3567,Fushë Kosovë,Fushë Kosovë,po,True
gjakove,Gjakovë / Đakovica,526,Gjakovë,Gjakovë,po,True
gjilan,Gjilan / Gnjilane,263,Gjilan,Gjilan,po,True
gllogoc,Gllogoc / Glogovac,140,Gllogoc,Gllogoc,po,True
gracanice,Graçanicë / Gračanica,14672,Graçanicë,Graçanicë,po,True
hani i elezit,Hani i Elezit / Elez Han,347,Hani i Elezit,Hani i Elezit,po,True



Rezultati i kontrollit të bashkimit


,vlera,pritet,kalon
rreshta para,46831,46831,True
rreshta pas,46831,46831,True
kolona para,42,42,True
kolona pas,50,50,True
komuna në dataset,38,38,True
komuna te municipalities.csv,38,38,True
komuna që përputhen,38,38,True
rreshta pa përputhje,0,0,True
rreshta me popullsi bosh pas bashkimit,0,0,True
rreshta me sipërfaqe njësie bosh pas bashkimit,0,0,True



processed\biodiversity_municipalities.csv: 46,831 rreshta, 50 kolona (42 origjinale + 8 të reja), 24.7 MB
Kolonat e reja: komuna_key, popullsia_2024, popullsia_e_regjistruar_2024, popullsia_vleresim_ask, siperfaqja_km2, njesia_e_siperfaqes, siperfaqja_njesise_km2, dendesia_banore_km2


## 6. AGREGIMI: KOMUNA DHE GRUPI TAKSONOMIK

Për çdo komunë: regjistrimet, speciet unike, speciet e kërcënuara (VU, EN, CR), specie për km² dhe regjistrime për 1,000 banorë. Krijohet edhe matrica **komunë × taxon_group**, input për PCA-në e Jetës (23 tetor).

`taxon_group` dhe `is_threatened` merren nga funksioni `add_taxon_features()` i Jetës (`03_vetite.ipynb`, 15 tetor), që të dy notebook-ët të përdorin të njëjtin definicion. Përshkrimi është te `reports/agregimi_sipas_komunes.md`.

In [19]:
section("6. AGREGIMI: KOMUNA DHE GRUPI TAKSONOMIK")
# Burimi: dataset-i i bashkuar me sipërfaqen dhe popullsinë (seksioni 5.2)
agg_src = df.assign(**{c: biodiversity_municipalities[c].values for c in
                       ["komuna_key", "popullsia_2024", "popullsia_vleresim_ask", "njesia_e_siperfaqes",
                        "siperfaqja_njesise_km2"]})
agg_src["komuna"] = alb(df_raw.municipality)
agg_src["rajoni"] = alb(df_raw.district)


# --- taxon_group dhe is_threatened nga funksioni i Jetës (03_vetite.ipynb, add_taxon_features) ---
# Funksioni lexohet nga notebook-u i saj pa e ndryshuar atë, që të dy notebook-ët të përdorin të njëjtin definicion.
import nbformat
_nb03 = nbformat.read(ROOT / "notebooks" / "03_vetite.ipynb", as_version=4)
_ns = {}
exec(next(cell.source for cell in _nb03.cells
          if cell.cell_type == "code" and "def add_taxon_features" in cell.source), _ns)
add_taxon_features = _ns["add_taxon_features"]
taxon_features = add_taxon_features(df_raw[["class", "iucnRedListCategory"]])
agg_src["taxon_group"] = taxon_features.taxon_group.values
agg_src["is_threatened"] = taxon_features.is_threatened.eq(1).fillna(False).astype(bool).values

GROUPS = ["Zogj", "Insekte", "Bimë", "Kërpudha", "Tjera", "E panjohur"]

group_map = (agg_src.groupby("taxon_group")
             .agg(regjistrime=("gbifID", "size"), klasa=("class", lambda s: s.nunique()),
                  klasat_kryesore=("class", lambda s: ", ".join(s.dropna().astype(str).value_counts().index[:3])))
             .reindex(GROUPS))
group_map.index.name = "taxon_group"
print("taxon_group (mapping-u sipas planit)")
display(group_map)

# --- Agregimi sipas komunës ---
g = agg_src.groupby("komuna")
agg_municipality = pd.DataFrame({
    "rajoni": g.rajoni.first(),
    "regjistrime": g.size(),
    "specie_unike": g.species.nunique(),
    "specie_te_kercenuara": agg_src[agg_src.is_threatened].groupby("komuna").species.nunique(),
    "regjistrime_te_kercenuara": agg_src[agg_src.is_threatened].groupby("komuna").size(),
    "popullsia_2024": g.popullsia_2024.first(),
    "popullsia_vleresim_ask": g.popullsia_vleresim_ask.first(),
    "njesia_e_siperfaqes": g.njesia_e_siperfaqes.first(),
    "siperfaqja_njesise_km2": g.siperfaqja_njesise_km2.first(),
}).fillna({"specie_te_kercenuara": 0, "regjistrime_te_kercenuara": 0})
agg_municipality[["specie_te_kercenuara", "regjistrime_te_kercenuara"]] = (
    agg_municipality[["specie_te_kercenuara", "regjistrime_te_kercenuara"]].astype(int))

# Për km² llogaritet sipas njësisë së sipërfaqes (dy Mitrovicat bashkë), që speciet të mos numërohen dy herë
unit_species = agg_src.groupby("njesia_e_siperfaqes").species.nunique()
unit_area = agg_src.groupby("njesia_e_siperfaqes").siperfaqja_njesise_km2.first()
agg_municipality["specie_per_km2"] = agg_municipality.njesia_e_siperfaqes.map(unit_species / unit_area).round(3)
agg_municipality["regjistrime_per_1000_banore"] = (
    agg_municipality.regjistrime / agg_municipality.popullsia_2024 * 1000).round(2)
agg_municipality["perqindja_e_regjistrimeve"] = (agg_municipality.regjistrime / len(agg_src) * 100).round(2)
agg_municipality = agg_municipality.sort_values("regjistrime", ascending=False)
agg_municipality.index.name = "komuna"

# --- Matrica komunë × taxon_group (input për PCA-në e Jetës) ---
records_matrix = pd.crosstab(agg_src.komuna, agg_src.taxon_group).reindex(columns=GROUPS, fill_value=0)
species_matrix = (agg_src.groupby(["komuna", "taxon_group"]).species.nunique().unstack(fill_value=0)
                  .reindex(columns=GROUPS, fill_value=0))
agg_municipality_taxon = pd.concat([records_matrix.add_prefix("regjistrime_"),
                                    species_matrix.add_prefix("specie_")], axis=1)
agg_municipality_taxon.insert(0, "rajoni", agg_municipality.rajoni)
agg_municipality_taxon = agg_municipality_taxon.loc[agg_municipality.index]
agg_municipality_taxon.index.name = "komuna"

# --- Kontrollet ---
rec_cols = [f"regjistrime_{x}" for x in GROUPS]
agg_check = pd.Series({
    "38 komuna në të dyja tabelat": len(agg_municipality) == 38 and len(agg_municipality_taxon) == 38,
    "regjistrimet mblidhen në 46,831": agg_municipality.regjistrime.sum() == len(df_raw),
    "matrica: shuma e rreshtit = regjistrimet e komunës": (
        agg_municipality_taxon[rec_cols].sum(axis=1) == agg_municipality.regjistrime).all(),
    "matrica mblidhet në 46,831": int(agg_municipality_taxon[rec_cols].to_numpy().sum()) == len(df_raw),
    "speciet e kërcënuara ≤ speciet unike": (agg_municipality.specie_te_kercenuara
                                            <= agg_municipality.specie_unike).all(),
    "pa vlera bosh te metrikat": agg_municipality[["specie_per_km2", "regjistrime_per_1000_banore"]]
                                 .notna().all().all(),
}, name="kalon")
display(agg_check.to_frame())
assert agg_check.all()

print("\nAgregimi sipas komunës (processed/agg_municipality.csv)")
display(agg_municipality)
print("Matrica komunë × taxon_group (processed/agg_municipality_taxon.csv)")
display(agg_municipality_taxon)

agg_municipality.to_csv(PROCESSED_DIR / "agg_municipality.csv")
agg_municipality_taxon.to_csv(PROCESSED_DIR / "agg_municipality_taxon.csv")
group_map.to_csv(REPORTS_DIR / "01_hartezimi_i_grupeve_taksonomike.csv")
print(f"\nprocessed/agg_municipality.csv: {agg_municipality.shape}  |  "
      f"processed/agg_municipality_taxon.csv: {agg_municipality_taxon.shape}")


6. AGREGIMI: KOMUNA DHE GRUPI TAKSONOMIK


taxon_group (mapping-u sipas planit)


,regjistrime,klasa,klasat_kryesore
taxon_group,,,
Zogj,28785,1,Aves
Insekte,11543,1,Insecta
Bimë,3738,11,"Magnoliopsida, Liliopsida, Polypodiopsida"
Kërpudha,971,22,"Lecanoromycetes, Agaricomycetes, Eurotiomycetes"
Tjera,1406,33,"Arachnida, Gastropoda, Mammalia"
E panjohur,388,0,


,kalon
38 komuna në të dyja tabelat,True
"regjistrimet mblidhen në 46,831",True
matrica: shuma e rreshtit = regjistrimet e komunës,True
"matrica mblidhet në 46,831",True
speciet e kërcënuara ≤ speciet unike,True
pa vlera bosh te metrikat,True



Agregimi sipas komunës (processed/agg_municipality.csv)


,rajoni,regjistrime,specie_unike,specie_te_kercenuara,regjistrime_te_kercenuara,popullsia_2024,popullsia_vleresim_ask,njesia_e_siperfaqes,siperfaqja_njesise_km2,specie_per_km2,regjistrime_per_1000_banore,perqindja_e_regjistrimeve
komuna,,,,,,,,,,,,
Graçanicë,Prishtinë,14672,334,3,366,18486,False,Graçanicë,122.41,2.729,793.68,31.33
Prishtinë,Prishtinë,5924,1149,13,38,227466,False,Prishtinë,523.13,2.196,26.04,12.65
Prizren,Prizren,3993,1243,12,31,147246,False,Prizren,626.86,1.983,27.12,8.53
Fushë Kosovë,Prishtinë,3567,180,4,81,63949,False,Fushë Kosovë,84.09,2.141,55.78,7.62
Kamenicë,Gjilan,2724,792,4,11,22868,False,Kamenicë,416.61,1.901,119.12,5.82
Pejë,Pejë,2449,1049,9,18,82745,False,Pejë,602.63,1.741,29.60,5.23
Leposaviq,Mitrovicë,1625,647,3,3,9485,True,Leposaviq,539.05,1.200,171.32,3.47
Deçan,Gjakovë,1403,729,5,8,27775,False,Deçan,293.97,2.480,50.51,3.00
Shtërpcë,Ferizaj,1160,582,7,18,10771,False,Shtërpcë,247.70,2.350,107.70,2.48


Matrica komunë × taxon_group (processed/agg_municipality_taxon.csv)


taxon_group,rajoni,regjistrime_Zogj,regjistrime_Insekte,regjistrime_Bimë,regjistrime_Kërpudha,regjistrime_Tjera,regjistrime_E panjohur,specie_Zogj,specie_Insekte,specie_Bimë,specie_Kërpudha,specie_Tjera,specie_E panjohur
komuna,,,,,,,,,,,,,
Graçanicë,Prishtinë,14460,129,57,0,26,0,191,85,40,0,18,0
Prishtinë,Prishtinë,3806,1009,846,51,209,3,140,552,356,40,58,3
Prizren,Prizren,1437,1734,563,14,186,59,170,683,276,9,75,30
Fushë Kosovë,Prishtinë,3496,16,21,0,34,0,142,14,14,0,10,0
Kamenicë,Gjilan,532,2118,38,3,33,0,98,649,26,2,17,0
Pejë,Pejë,766,892,427,104,204,56,101,576,252,40,75,5
Leposaviq,Mitrovicë,166,1320,59,0,38,42,84,489,27,0,13,34
Deçan,Gjakovë,289,512,173,343,66,20,68,366,101,175,19,0
Shtërpcë,Ferizaj,290,630,180,13,47,0,95,347,108,3,29,0



processed/agg_municipality.csv: (38, 12)  |  processed/agg_municipality_taxon.csv: (38, 13)


## 6.2 AGREGIMI SIPAS KOHËS

Regjistrimet dhe speciet unike për vit, dhe tabela **vit × muaj** (regjistrime). Të dyja janë në `processed/agg_time.csv`: një rresht për vit, nga viti i parë deri te i fundit, edhe vitet pa regjistrime (0). Kolona `pa_muaj` i numëron rreshtat që kanë vit, por jo muaj (data vetëm me vit).

Përdoren `year` dhe `month` origjinale. Kontrolli i Delizës (19 tetor) është përfunduar me 0 plotësime të lejueshme sipas rregullit konservativ, prandaj këto vlera përputhen me kopjen aktuale të pastruar. Shih `reports/plotesimi_year_month_para_pas.csv`. Agregimi ri-ekzekutohet mbi dataset-in final pas pastrimit të mëvonshëm. Përshkrimi është te `reports/agregimi_sipas_kohes.md`.

In [20]:
section("6.2 AGREGIMI SIPAS KOHËS")
# Kontrolli i Delizës përfundoi me 0 plotësime; year/month mbeten si në origjinal.
MONTHS = list(range(1, 13))
MONTH_NAMES = ["jan", "shk", "mar", "pri", "maj", "qer", "kor", "gus", "sht", "tet", "nën", "dhj"]
t_src = df[df.year.notna()].copy()
# Përqendrimi më 30 nëntor sugjeron placeholder të mundshëm; kërkohet konfirmim nga burimi.
# Flamuri kontrollon vetëm publikuesin dhe datën, jo origjinën nga literatura; datat nuk ndryshohen.
t_src["data_30_nentor_habiprot"] = ((publisher[t_src.index] == "HabiProt") & (t_src.month == 11) & (t_src.day == 30))

# --- Regjistrimet dhe speciet unike për vit (të gjitha vitet nga min deri max, edhe ato pa regjistrime) ---
years = pd.RangeIndex(int(t_src.year.min()), int(t_src.year.max()) + 1, name="viti")
gy = t_src.groupby(t_src.year.astype(int))
agg_year = pd.DataFrame({"regjistrime": gy.size(), "specie_unike": gy.species.nunique(),
                         "data_30_nentor_habiprot": gy.data_30_nentor_habiprot.sum()}).reindex(years, fill_value=0)
agg_year["perqindja_e_regjistrimeve"] = (agg_year.regjistrime / len(t_src) * 100).round(3)

# --- Tabela vit × muaj (regjistrime); rreshtat me vit, por pa muaj, në kolonë të veçantë ---
ym = pd.crosstab(t_src.year.astype(int), t_src.month.fillna(0).astype(int)).reindex(index=years, fill_value=0)
ym = ym.reindex(columns=[0] + MONTHS, fill_value=0)
ym.columns = ["pa_muaj"] + [f"{m:02d}_{n}" for m, n in zip(MONTHS, MONTH_NAMES)]
agg_time = pd.concat([agg_year, ym[[c for c in ym.columns if c != "pa_muaj"] + ["pa_muaj"]]], axis=1)

# --- Kontrollet ---
month_cols = [c for c in agg_time.columns if c[:2].isdigit()]
time_check = pd.Series({
    "regjistrimet me vit mblidhen në numrin e rreshtave me vit": agg_time.regjistrime.sum() == len(t_src),
    "shuma e muajve + pa_muaj = regjistrimet e vitit": (
        agg_time[month_cols + ["pa_muaj"]].sum(axis=1) == agg_time.regjistrime).all(),
    "vitet pa regjistrime kanë 0 specie": (agg_time.loc[agg_time.regjistrime == 0, "specie_unike"] == 0).all(),
    "month vetëm 1–12": t_src.month.dropna().between(1, 12).all(),
}, name="kalon")
display(time_check.to_frame())
assert time_check.all()

n_years_with = int((agg_time.regjistrime > 0).sum())
print(f"Vitet {years[0]}–{years[-1]}: {len(years)} vite, {n_years_with} me regjistrime  |  "
      f"rreshta me vit: {len(t_src):,}  |  pa vit (jashtë agregimit): {int(df.year.isna().sum()):,}  |  "
      f"me vit, por pa muaj: {int(agg_time.pa_muaj.sum()):,}")
print(f"Regjistrime HabiProt me datë 30 nëntor, për shqyrtim: {int(t_src.data_30_nentor_habiprot.sum()):,} "
      f"(placeholder i mundshëm, jo i konfirmuar); pa to, nëntori ka "
      f"{int(((t_src.month == 11) & ~t_src.data_30_nentor_habiprot).sum()):,} regjistrime.")

# Përmbledhje sipas dekadës dhe sipas muajit (vetëm për lexim)
decade = (t_src.year.astype(int) // 10 * 10)
by_decade = pd.DataFrame({"regjistrime": t_src.groupby(decade).size(),
                          "specie_unike": t_src.groupby(decade).species.nunique()})
by_decade.index = [f"{d}–{d + 9}" for d in by_decade.index]
by_decade.index.name = "dekada"
by_month = pd.DataFrame({"regjistrime": agg_time[month_cols].sum(),
                         "specie_unike": t_src.groupby(t_src.month).species.nunique().reindex(MONTHS).values})
by_month["perqindja"] = (by_month.regjistrime / by_month.regjistrime.sum() * 100).round(2)
clean_month = t_src[~t_src.data_30_nentor_habiprot].groupby("month").size().reindex(MONTHS, fill_value=0)
by_month["regjistrime_pa_30_nentor"] = clean_month.values
by_month["perqindja_pa_30_nentor"] = (by_month.regjistrime_pa_30_nentor / clean_month.sum() * 100).round(2)
by_month.index.name = "muaji"

print("\nRegjistrimet dhe speciet unike për vit (vetëm vitet me regjistrime; CSV-ja i ka të gjitha vitet)")
display(agg_time.loc[agg_time.regjistrime > 0, ["regjistrime", "specie_unike", "perqindja_e_regjistrimeve",
                                                 "data_30_nentor_habiprot"]])
print("Tabela vit × muaj (regjistrime), 2000 e tutje")
display(agg_time.loc[2000:, month_cols + ["pa_muaj"]])
print("Përmbledhje sipas dekadës")
display(by_decade)
print("Përmbledhje sipas muajit (të gjitha vitet)")
display(by_month)

agg_time.to_csv(PROCESSED_DIR / "agg_time.csv")
by_decade.to_csv(REPORTS_DIR / "01_koha_sipas_dekades.csv")
by_month.to_csv(REPORTS_DIR / "01_koha_sipas_muajit.csv")
print(f"\nprocessed/agg_time.csv: {agg_time.shape[0]} vite × {agg_time.shape[1]} kolona "
      f"(regjistrime, specie_unike, %, data_30_nentor_habiprot, 12 muaj, pa_muaj)")


6.2 AGREGIMI SIPAS KOHËS


,kalon
regjistrimet me vit mblidhen në numrin e rreshtave me vit,True
shuma e muajve + pa_muaj = regjistrimet e vitit,True
vitet pa regjistrime kanë 0 specie,True
month vetëm 1–12,True


Vitet 1826–2026: 201 vite, 102 me regjistrime  |  rreshta me vit: 44,563  |  pa vit (jashtë agregimit): 2,268  |  me vit, por pa muaj: 599
Regjistrime HabiProt me datë 30 nëntor, për shqyrtim: 3,598 (placeholder i mundshëm, jo i konfirmuar); pa to, nëntori ka 1,191 regjistrime.

Regjistrimet dhe speciet unike për vit (vetëm vitet me regjistrime; CSV-ja i ka të gjitha vitet)


,regjistrime,specie_unike,perqindja_e_regjistrimeve,data_30_nentor_habiprot
viti,,,,
1826,1,1,0.002,0
1835,1,1,0.002,0
1838,1,1,0.002,0
1860,1,1,0.002,0
1884,1,0,0.002,0
...,...,...,...,...
2022,5091,724,11.424,0
2023,1947,363,4.369,0
2024,3758,773,8.433,0


Tabela vit × muaj (regjistrime), 2000 e tutje


...

Përmbledhje sipas dekadës


,regjistrime,specie_unike
dekada,,
1820–1829,1,1
1830–1839,2,2
1860–1869,1,1
1880–1889,1,0
1890–1899,11,10
1900–1909,124,20
1910–1919,113,46
1920–1929,6,3
1930–1939,109,73


Përmbledhje sipas muajit (të gjitha vitet)


,regjistrime,specie_unike,perqindja,regjistrime_pa_30_nentor,perqindja_pa_30_nentor
muaji,,,,,
01_jan,1024,127,2.33,1024,2.54
02_shk,1924,182,4.38,1924,4.77
03_mar,3586,288,8.16,3586,8.88
04_pri,4598,598,10.46,4598,11.39
05_maj,6353,1066,14.45,6353,15.74
06_qer,5600,988,12.74,5600,13.87
07_kor,5263,1157,11.97,5263,13.04
08_gus,5146,1028,11.71,5146,12.75
09_sht,3296,700,7.50,3296,8.17



processed/agg_time.csv: 201 vite × 17 kolona (regjistrime, specie_unike, %, data_30_nentor_habiprot, 12 muaj, pa_muaj)


## 6.3 AGREGIMI: SPECIET DHE RAJONI

- **Tabela klasë × rajon:** regjistrimet dhe speciet unike të çdo klase në 7 rajonet → `processed/agg_class_district.csv`.
- **Tabela në nivel specieje:** për secilën nga 4,095 speciet, viti i parë dhe i fundit, numri i komunave dhe numri i regjistrimeve → `processed/agg_species.csv`.

Përshkrimi është te `reports/agregimi_sipas_specieve_dhe_rajonit.md`.

In [21]:
section("6.3 AGREGIMI: SPECIET DHE RAJONI")
sp_src = pd.DataFrame({
    "species": df_raw.species, "kingdom": df_raw.kingdom, "class": df_raw["class"], "family": df_raw.family,
    "iucnRedListCategory": df_raw.iucnRedListCategory, "year": df.year, "komuna": agg_src.komuna,
    "rajoni": agg_src.rajoni, "taxon_group": agg_src.taxon_group, "is_threatened": agg_src.is_threatened,
})
DISTRICTS = sorted(sp_src.rajoni.unique())

# --- 1. Tabela klasë × rajon (regjistrime dhe specie unike) ---
klasa = sp_src["class"].fillna("(mungon)")
rec_cd = pd.crosstab(klasa, sp_src.rajoni).reindex(columns=DISTRICTS, fill_value=0)
spc_cd = (sp_src.groupby([klasa, "rajoni"]).species.nunique().unstack(fill_value=0)
          .reindex(index=rec_cd.index, columns=DISTRICTS, fill_value=0))
agg_class_district = pd.concat([rec_cd.add_prefix("regjistrime_"), spc_cd.add_prefix("specie_")], axis=1)
agg_class_district.insert(0, "taxon_group", sp_src.groupby(klasa).taxon_group.first())
agg_class_district.insert(1, "regjistrime_gjithsej", rec_cd.sum(axis=1))
agg_class_district.insert(2, "specie_gjithsej", sp_src.groupby(klasa).species.nunique())
agg_class_district = agg_class_district.sort_values("regjistrime_gjithsej", ascending=False)
agg_class_district.index.name = "klasa"

# --- 2. Tabela në nivel specieje ---
with_sp = sp_src[sp_src.species.notna()]
gs = with_sp.groupby("species")
agg_species = pd.DataFrame({
    "kingdom": gs.kingdom.first(), "class": gs["class"].first(), "family": gs.family.first(),
    "taxon_group": gs.taxon_group.first(), "iucnRedListCategory": gs.iucnRedListCategory.first(),
    "regjistrime": gs.size(),
    "viti_i_pare": gs.year.min(), "viti_i_fundit": gs.year.max(),
    "vite_me_regjistrime": gs.year.nunique(),
    "regjistrime_pa_vit": gs.year.apply(lambda s: int(s.isna().sum())),
    "komuna": gs.komuna.nunique(), "rajone": gs.rajoni.nunique(),
}).sort_values("regjistrime", ascending=False)
agg_species.index.name = "species"

# --- Kontrollet ---
species_check = pd.Series({
    "klasë × rajon mblidhet në 46,831": int(rec_cd.to_numpy().sum()) == len(df_raw),
    "7 rajone": len(DISTRICTS) == 7,
    "4,095 specie në tabelën e specieve": len(agg_species) == df_raw.species.nunique(),
    "regjistrimet e specieve mblidhen në rreshtat me species": agg_species.regjistrime.sum() == with_sp.shape[0],
    "viti i parë ≤ viti i fundit": (agg_species.dropna(subset=["viti_i_pare"]).viti_i_pare
                                    <= agg_species.dropna(subset=["viti_i_pare"]).viti_i_fundit).all(),
    "komuna 1–38, rajone 1–7": agg_species.komuna.between(1, 38).all() and agg_species.rajone.between(1, 7).all(),
    "çdo specie ka një klasë dhe një kategori IUCN": (gs["class"].nunique() <= 1).all()
                                                    and (gs.iucnRedListCategory.nunique() <= 1).all(),
}, name="kalon")
display(species_check.to_frame())
assert species_check.all()

# --- Përmbledhje ---
species_summary = pd.Series({
    "specie gjithsej": len(agg_species),
    "specie me 1 regjistrim (singletons)": int((agg_species.regjistrime == 1).sum()),
    "specie në vetëm 1 komunë": int((agg_species.komuna == 1).sum()),
    "specie në të 7 rajonet": int((agg_species.rajone == 7).sum()),
    "specie pa asnjë vit": int(agg_species.viti_i_pare.isna().sum()),
    "specie të parë për herë të fundit para 2000": int((agg_species.viti_i_fundit < 2000).sum()),
    "specie të parë për herë të fundit para 1950": int((agg_species.viti_i_fundit < 1950).sum()),
    "specie të reja (viti i parë ≥ 2020)": int((agg_species.viti_i_pare >= 2020).sum()),
}, name="vlera").to_frame()
threatened_species = agg_species[agg_species.iucnRedListCategory.isin(["VU", "EN", "CR"])].sort_values(
    ["iucnRedListCategory", "regjistrime"], ascending=[True, False])

print("Tabela klasë × rajon (15 klasat me më shumë regjistrime; CSV-ja i ka të gjitha)")
display(agg_class_district.head(15))
print("Pjesa e klasave brenda çdo rajoni (%), 8 klasat kryesore")
display((rec_cd.div(rec_cd.sum()) * 100).round(2).loc[agg_class_district.index[:8]])
print("\nTabela në nivel specieje (20 speciet me më shumë regjistrime; CSV-ja i ka të 4,095)")
display(agg_species.head(20))
print("Përmbledhje")
display(species_summary)
print(f"Speciet e kërcënuara (VU, EN, CR): {len(threatened_species)}")
display(threatened_species)

agg_class_district.to_csv(PROCESSED_DIR / "agg_class_district.csv")
agg_species.to_csv(PROCESSED_DIR / "agg_species.csv")
species_summary.to_csv(REPORTS_DIR / "01_specie_permbledhje.csv")
threatened_species.to_csv(REPORTS_DIR / "01_specie_te_kercenuara.csv")
print(f"\nprocessed/agg_class_district.csv: {agg_class_district.shape}  |  "
      f"processed/agg_species.csv: {agg_species.shape}")


6.3 AGREGIMI: SPECIET DHE RAJONI


,kalon
"klasë × rajon mblidhet në 46,831",True
7 rajone,True
"4,095 specie në tabelën e specieve",True
regjistrimet e specieve mblidhen në rreshtat me species,True
viti i parë ≤ viti i fundit,True
"komuna 1–38, rajone 1–7",True
çdo specie ka një klasë dhe një kategori IUCN,True


Tabela klasë × rajon (15 klasat me më shumë regjistrime; CSV-ja i ka të gjitha)


rajoni,taxon_group,regjistrime_gjithsej,specie_gjithsej,regjistrime_Ferizaj,regjistrime_Gjakovë,regjistrime_Gjilan,regjistrime_Mitrovicë,regjistrime_Pejë,regjistrime_Prishtinë,regjistrime_Prizren,specie_Ferizaj,specie_Gjakovë,specie_Gjilan,specie_Mitrovicë,specie_Pejë,specie_Prishtinë,specie_Prizren
klasa,,,,,,,,,,,,,,,,,
Aves,Zogj,28785,255,772,747,1403,646,1064,22607,1546,140,116,142,146,119,218,170
Insecta,Insekte,11543,2226,1273,874,2526,1945,1054,1548,2323,509,480,709,682,630,673,791
Magnoliopsida,Bimë,2701,760,277,284,79,101,454,912,594,175,162,64,74,263,374,287
Liliopsida,Bimë,940,143,68,99,79,207,78,266,143,32,32,19,23,36,79,43
Lecanoromycetes,Kërpudha,612,238,1,306,0,2,32,7,264,1,152,0,1,24,5,127
(mungon),E panjohur,388,63,4,58,8,154,85,9,70,3,19,5,37,16,7,36
Arachnida,Tjera,358,63,13,33,24,22,107,70,89,9,13,11,14,24,20,33
Gastropoda,Tjera,279,48,76,19,3,11,72,72,26,10,8,3,8,24,16,7
Mammalia,Tjera,273,51,43,13,11,23,38,69,76,21,12,7,16,23,27,28


Pjesa e klasave brenda çdo rajoni (%), 8 klasat kryesore


...

Përmbledhje


,vlera
specie gjithsej,4095
specie me 1 regjistrim (singletons),1727
specie në vetëm 1 komunë,2155
specie në të 7 rajonet,156
specie pa asnjë vit,416
specie të parë për herë të fundit para 2000,754
specie të parë për herë të fundit para 1950,68
specie të reja (viti i parë ≥ 2020),779


Speciet e kërcënuara (VU, EN, CR): 37


,kingdom,class,family,taxon_group,iucnRedListCategory,regjistrime,viti_i_pare,viti_i_fundit,vite_me_regjistrime,regjistrime_pa_vit,komuna,rajone
species,,,,,,,,,,,,
Anguilla anguilla,Animalia,NaN,Anguillidae,E panjohur,CR,3,1992,1992,1,2,3,3
Cheilosia balkana,Animalia,Insecta,Syrphidae,Insekte,EN,11,1997,2016,2,0,1,1
Schistostege decussata,Animalia,Insecta,Geometridae,Insekte,EN,2,1979,1979,1,1,2,2
Alburnoides prespensis,Animalia,NaN,Cyprinidae,E panjohur,EN,2,1985,1992,2,0,1,1
Phoxinus strymonicus,Animalia,NaN,Cyprinidae,E panjohur,EN,1,2010,2010,1,0,1,1
Pipizella bispina,Animalia,Insecta,Syrphidae,Insekte,EN,1,2016,2016,1,0,1,1
Anser cygnoides,Animalia,Aves,Anatidae,Zogj,EN,1,2025,2025,1,0,1,1
Aythya ferina,Animalia,Aves,Anatidae,Zogj,VU,382,2001,2026,12,0,4,2
Streptopelia turtur,Animalia,Aves,Columbidae,Zogj,VU,110,1984,2026,14,3,22,7



processed/agg_class_district.csv: (69, 17)  |  processed/agg_species.csv: (4095, 12)


## 7. MOSTRIMI

Dy mostra me 10,000 rreshta dhe `random_state = 42`:

- **e rastësishme:** `df.sample(n=10_000)`;
- **e shtresëzuar sipas klasës:** çdo klasë merr numër rreshtash proporcional me pjesën e saj në dataset (rrumbullakim me mbetjen më të madhe, që totali të jetë saktësisht 10,000). Rreshtat pa `class` janë shtresë më vete.

Krahasohet % sipas klasës: dataset-i i plotë kundrejt secilës mostër. Mostrat ruhen me 42 kolonat origjinale te `processed/sample_random.csv` dhe `processed/sample_stratified.csv`. Përshkrimi, me vendimin dhe arsyetimin, është te `reports/mostrimi.md`.

In [22]:
section("7. MOSTRIMI")
SAMPLE_N = 10_000
RANDOM_STATE = 42
strata = df_raw["class"].fillna("(mungon)")  # shtresat: klasa; mungesa e klasës është shtresë më vete

# --- 1. Mostra e rastësishme ---
random_idx = df_raw.sample(n=SAMPLE_N, random_state=RANDOM_STATE).index.sort_values()


# --- 2. Mostra e shtresëzuar sipas klasës: shpërndarje proporcionale, rrumbullakim me mbetjen më të madhe ---
def proportional_allocation(groups, n):
    counts = groups.value_counts().sort_index()
    exact = counts / counts.sum() * n
    alloc = np.floor(exact).astype(int)
    remainder = (exact - alloc).sort_values(ascending=False, kind="stable")
    alloc[remainder.index[: n - alloc.sum()]] += 1
    return alloc


allocation = proportional_allocation(strata, SAMPLE_N)
stratified_idx = pd.Index(np.concatenate([
    strata[strata == k].sample(n=n, random_state=RANDOM_STATE).index.to_numpy()
    for k, n in allocation.items() if n > 0
])).sort_values()

# Mostrat ruhen me 42 kolonat ashtu siç janë në CSV-në burimore (raw_text nga seksioni 5.2)
sample_random = raw_text.loc[random_idx]
sample_stratified = raw_text.loc[stratified_idx]


# --- 3. Krahasimi i % sipas klasës: dataset-i i plotë kundrejt secilës mostër ---
def pct(index, by):
    return by.loc[index].value_counts(normalize=True) * 100


cmp_class = pd.DataFrame({
    "regjistrime_plote": strata.value_counts(),
    "perqindja_plote": pct(df_raw.index, strata),
    "perqindja_rastesishme": pct(random_idx, strata),
    "perqindja_shtresezuar": pct(stratified_idx, strata),
}).fillna(0).sort_values("regjistrime_plote", ascending=False)
cmp_class["regjistrime_plote"] = cmp_class.regjistrime_plote.astype(int)
cmp_class["n_rastesishme"] = strata.loc[random_idx].value_counts().reindex(cmp_class.index, fill_value=0)
cmp_class["n_shtresezuar"] = strata.loc[stratified_idx].value_counts().reindex(cmp_class.index, fill_value=0)
cmp_class["diferenca_rastesishme"] = cmp_class.perqindja_rastesishme - cmp_class.perqindja_plote
cmp_class["diferenca_shtresezuar"] = cmp_class.perqindja_shtresezuar - cmp_class.perqindja_plote
cmp_class = cmp_class.round(3)
cmp_class.index.name = "klasa"


def tvd(p, q):
    """Distanca e variacionit total (0 = shpërndarje identike, 100 = krejt të ndryshme), në pikë përqindjeje."""
    return 0.5 * (p - q).abs().sum()


# Sa ndryshon mostra e rastësishme vetëm nga rastësia: 200 farë të ndryshme
seed_tvd = pd.Series([tvd(pct(df_raw.sample(n=SAMPLE_N, random_state=s).index, strata).reindex(cmp_class.index,
                                                                                                fill_value=0),
                          cmp_class.perqindja_plote) for s in range(200)])
n_classes = strata.nunique()
sampling_summary = pd.DataFrame({
    "rreshta": [len(df_raw), len(random_idx), len(stratified_idx)],
    "klasa_te_perfaqesuara": [n_classes, strata.loc[random_idx].nunique(), strata.loc[stratified_idx].nunique()],
    "diferenca_max_pp": [0, cmp_class.diferenca_rastesishme.abs().max(), cmp_class.diferenca_shtresezuar.abs().max()],
    "tvd_pp": [0, tvd(cmp_class.perqindja_rastesishme, cmp_class.perqindja_plote),
               tvd(cmp_class.perqindja_shtresezuar, cmp_class.perqindja_plote)],
}, index=["dataset-i i plotë", "mostra e rastësishme", "mostra e shtresëzuar"]).round(3)
sampling_summary.index.name = "mostra"

# Kontroll dytësor: rajoni (nuk është variabël shtresëzimi)
district_alb = alb(df_raw.district)
cmp_district = pd.DataFrame({
    "perqindja_plote": pct(df_raw.index, district_alb),
    "perqindja_rastesishme": pct(random_idx, district_alb),
    "perqindja_shtresezuar": pct(stratified_idx, district_alb),
}).round(2).sort_values("perqindja_plote", ascending=False)
cmp_district.index.name = "rajoni"

# --- Kontrollet ---
sample_check = pd.Series({
    "mostra e rastësishme ka 10,000 rreshta": len(sample_random) == SAMPLE_N,
    "mostra e shtresëzuar ka 10,000 rreshta": len(sample_stratified) == SAMPLE_N,
    "pa rreshta të përsëritur në mostra": random_idx.is_unique and stratified_idx.is_unique,
    "shpërndarja proporcionale mblidhet në 10,000": allocation.sum() == SAMPLE_N,
    "çdo shtresë merr ±1 nga pjesa e saj e saktë": ((allocation - strata.value_counts().sort_index()
                                                    / len(df_raw) * SAMPLE_N).abs() < 1).all(),
    "e njëjta farë jep të njëjtën mostër": df_raw.sample(n=SAMPLE_N, random_state=RANDOM_STATE).index
                                           .sort_values().equals(random_idx),
    "gbifID të mostrave ekzistojnë në dataset": sample_random.gbifID.isin(raw_text.gbifID).all()
                                                and sample_stratified.gbifID.isin(raw_text.gbifID).all(),
}, name="kalon")
display(sample_check.to_frame())
assert sample_check.all()

print(f"\nMostrat: n = {SAMPLE_N:,}, random_state = {RANDOM_STATE}, shtresat = {n_classes} (68 klasa + mungon)")
print("Përmbledhja e krahasimit")
display(sampling_summary)
print(f"Mostra e rastësishme me 200 farë: TVD mesatare {seed_tvd.mean():.2f} pp, "
      f"95% e farave nën {seed_tvd.quantile(0.95):.2f} pp (mostra e shtresëzuar: "
      f"{sampling_summary.loc['mostra e shtresëzuar', 'tvd_pp']:.3f} pp)")
print("\nKrahasimi i % sipas klasës (15 klasat kryesore; CSV-ja i ka të gjitha)")
display(cmp_class.head(15))
print("Kontroll dytësor: % sipas rajonit")
display(cmp_district)

sample_random.to_csv(PROCESSED_DIR / "sample_random.csv", index=False)
sample_stratified.to_csv(PROCESSED_DIR / "sample_stratified.csv", index=False)
cmp_class.to_csv(REPORTS_DIR / "01_mostrimi_krahasimi_klasa.csv")
sampling_summary.to_csv(REPORTS_DIR / "01_mostrimi_permbledhje.csv")
cmp_district.to_csv(REPORTS_DIR / "01_mostrimi_krahasimi_rajoni.csv")
print(f"\nprocessed/sample_random.csv: {sample_random.shape}  |  processed/sample_stratified.csv: "
      f"{sample_stratified.shape}")


7. MOSTRIMI


,kalon
"mostra e rastësishme ka 10,000 rreshta",True
"mostra e shtresëzuar ka 10,000 rreshta",True
pa rreshta të përsëritur në mostra,True
"shpërndarja proporcionale mblidhet në 10,000",True
çdo shtresë merr ±1 nga pjesa e saj e saktë,True
e njëjta farë jep të njëjtën mostër,True
gbifID të mostrave ekzistojnë në dataset,True



Mostrat: n = 10,000, random_state = 42, shtresat = 69 (68 klasa + mungon)
Përmbledhja e krahasimit


,rreshta,klasa_te_perfaqesuara,diferenca_max_pp,tvd_pp
mostra,,,,
dataset-i i plotë,46831,69,0.000,0.000
mostra e rastësishme,10000,46,0.222,0.703
mostra e shtresëzuar,10000,42,0.006,0.104


Mostra e rastësishme me 200 farë: TVD mesatare 0.89 pp, 95% e farave nën 1.36 pp (mostra e shtresëzuar: 0.104 pp)

Krahasimi i % sipas klasës (15 klasat kryesore; CSV-ja i ka të gjitha)


,regjistrime_plote,perqindja_plote,perqindja_rastesishme,perqindja_shtresezuar,n_rastesishme,n_shtresezuar,diferenca_rastesishme,diferenca_shtresezuar
klasa,,,,,,,,
Aves,28785,61.466,61.58,61.47,6158,6147,0.114,0.004
Insecta,11543,24.648,24.50,24.65,2450,2465,-0.148,0.002
Magnoliopsida,2701,5.768,5.99,5.77,599,577,0.222,0.002
Liliopsida,940,2.007,1.90,2.01,190,201,-0.107,0.003
Lecanoromycetes,612,1.307,1.28,1.31,128,131,-0.027,0.003
(mungon),388,0.829,0.78,0.83,78,83,-0.049,0.001
Arachnida,358,0.764,0.76,0.77,76,77,-0.004,0.006
Gastropoda,279,0.596,0.54,0.60,54,60,-0.056,0.004
Mammalia,273,0.583,0.67,0.58,67,58,0.087,-0.003


Kontroll dytësor: % sipas rajonit


,perqindja_plote,perqindja_rastesishme,perqindja_shtresezuar
rajoni,,,
Prishtinë,54.95,54.91,55.54
Prizren,11.25,11.53,11.01
Gjilan,8.87,9.33,8.60
Mitrovicë,6.82,6.72,7.03
Pejë,6.81,6.97,6.62
Gjakovë,5.76,5.44,5.74
Ferizaj,5.54,5.10,5.46



processed/sample_random.csv: (10000, 42)  |  processed/sample_stratified.csv: (10000, 42)
